# Unit 8. Generalised Linear Models II: Counts and the Exponential Family

精算統計模型 (SOA Exam SRM/PA). The text of [unit08.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit08.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("ISLR2", "MASS", "numDeriv", "pscl", "statmod", "tweedie")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/AutoCollision.csv", "data/frees/csv/HealthExpend.csv", "data/frees/csv/SingaporeAuto.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2)
suppressMessages(library(MASS))

## 1 Counts and the Poisson Distribution

> **Reading.** FREES §12.1.1; ISLR §4.6.1, §4.6.2 (PA reading list, not SRM). SRM learning outcomes 2(a), 2(b).

**Remark** (The classical assumptions; Unit 2). A1–A5 are Unit 2's classical assumptions on $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\varepsilon}$ with $\mathbf{X}$ fixed: linearity (A1), zero-mean errors (A2), $\operatorname{var}(\boldsymbol{\varepsilon})=\sigma^2\mathbf{I}_n$ with $\sigma^2>0$ (A3), $\operatorname{rank}(\mathbf{X})=k+1$, with $n>k+1$ for inference (A4), and normal errors (A5).

### 1.1 The Poisson distribution

**Definition 8.1** (Poisson; FREES §12.1.1, ISLR §4.6.2). A **count** takes values in $\{0,1,2,\ldots\}$. A count $y$ has the **Poisson** distribution with mean $\mu>0$, written $y\sim\mathrm{Pois}(\mu)$, if

$$
\operatorname{\mathsf{P}}(y=j)=\frac{\mu^j}{j!}\,e^{-\mu} , \qquad j=0,1,2,\ldots \tag{1}
$$

**Proposition 8.2** (Moments and additivity; FREES §12.1.1). Let $y\sim\mathrm{Pois}(\mu)$.

(i) $\operatorname{\mathsf{E}}[s^y]=e^{\mu(s-1)}$ for every $s\in\mathbb{R}$.

(ii) $\operatorname{\mathsf{E}} y=\mu$ and $\operatorname{var} y=\mu$ (**equidispersion**); every moment is finite, and $\operatorname{\mathsf{E}}(y-\mu)^3=\mu$, $\operatorname{\mathsf{E}}(y-\mu)^4=\mu+3\mu^2$.

(iii) If $y_1,\ldots,y_m$ are independent with $y_j\sim\mathrm{Pois}(\mu_j)$, then $\sum_jy_j\sim\mathrm{Pois}(\sum_j\mu_j)$.

**Proof.** (i) $\operatorname{\mathsf{E}}[s^y]=e^{-\mu}\sum_{j\geqslant0}(\mu s)^j/j!=e^{-\mu}e^{\mu s}$, the series converging absolutely for every $s$.

(ii) For $r\geqslant1$, $\operatorname{\mathsf{E}}[y(y-1)\cdots(y-r+1)]=e^{-\mu}\sum_{j\geqslant r} \mu^j/(j-r)!=\mu^r$, the terms with $j<r$ vanishing; hence every moment is a finite combination of these and is finite. With $r=1,2,3,4$: $\operatorname{\mathsf{E}} y=\mu$, $\operatorname{\mathsf{E}} y^2=\mu^2+\mu$, $\operatorname{\mathsf{E}} y^3=\mu^3+3\mu^2+\mu$ and $\operatorname{\mathsf{E}} y^4=\mu^4+6\mu^3+7\mu^2+\mu$, using $y^2=y(y-1)+y$, $y^3=y(y-1)(y-2)+3y(y-1)+y$ and $y^4=y(y-1)(y-2)(y-3)+6y(y-1)(y-2)+7y(y-1)+y$. Expanding the central moments binomially, $\operatorname{var} y=\mu$, $\operatorname{\mathsf{E}}(y-\mu)^3=\operatorname{\mathsf{E}} y^3-3\mu\operatorname{\mathsf{E}} y^2+2\mu^3=\mu$ and $\operatorname{\mathsf{E}}(y-\mu)^4=\operatorname{\mathsf{E}} y^4-4\mu\operatorname{\mathsf{E}} y^3+6\mu^2\operatorname{\mathsf{E}} y^2-3\mu^4=\mu+3\mu^2$.

(iii) By independence the generating function of the sum is $\prod_je^{\mu_j(s-1)}=e^{(\sum_j\mu_j)(s-1)}$, the generating function of $\mathrm{Pois}(\sum_j\mu_j)$. The law itself follows by direct convolution: for two independent summands, $\operatorname{\mathsf{P}}(y_1+y_2=t)=\sum_{a=0}^te^{-\mu_1}\frac{\mu_1^a}{a!} e^{-\mu_2}\frac{\mu_2^{t-a}}{(t-a)!}=\frac{e^{-(\mu_1+\mu_2)}}{t!}\sum_{a=0}^t\binom ta \mu_1^a\mu_2^{t-a}=e^{-(\mu_1+\mu_2)}\frac{(\mu_1+\mu_2)^t}{t!}$ by the binomial theorem, and induction on the number of summands gives the general case. ∎

**Proposition 8.3** (The sample mean is the maximum likelihood estimate; FREES §12.1.1). Let $y_1,\ldots,y_n$ be independent $\mathrm{Pois}(\mu)$ with $\bar y>0$. The log-likelihood

$$
L(\mu)=\sum_{i=1}^n\bigl(-\mu+y_i\ln\mu-\ln y_i!\bigr) , \qquad \mu>0, \tag{2}
$$

has the unique maximiser $\hat\mu=\bar y$. If $\bar y=0$ it has no maximiser on $(0,\infty)$.

**Proof.** $L'(\mu)=-n+n\bar y/\mu$ and $L''(\mu)=-n\bar y/\mu^2<0$ when $\bar y>0$, so $L$ is strictly concave (the prerequisites' lemma **convexity from the derivative**) with its only stationary point at $\bar y$, its unique maximiser by the prerequisites' proposition **first-order condition**. If $\bar y=0$ then $L(\mu)=-n\mu$ is strictly decreasing on $(0,\infty)$. ∎

**Example 8.1** (Singapore automobile claims; FREES §12.1.1, Table 12.1). FREES's $n=7483$ automobile policies of a Singapore insurer in 1993, with `Clm_Count` the number of accidents in the year:

In [ ]:
S <- read.csv("data/frees/csv/SingaporeAuto.csv", quote = "")
y <- S$Clm_Count; n <- length(y)
table(y); ybar <- mean(y); ybar
round(n * dpois(0:4, ybar), 2)

The mean is $\bar y=0.06989$, and by Proposition 8.3 the fitted counts $n\,\operatorname{\mathsf{P}}(y=j)$ at $\hat\mu=\bar y$ are $6977.86$, $487.69$ ($487.6948$ to four decimals; FREES prints $487.70$), $17.04$, $0.40$ and $0.01$, FREES's Table 12.1.

### 1.2 Pearson's goodness-of-fit statistic

**Definition 8.4** (Pearson's chi-square statistic; FREES (12.3)). Let $C_1,\ldots,C_c$ ($c\geqslant2$) partition $\{0,1,2,\ldots\}$ into **cells**, let $N_j=\#\{i:y_i\in C_j\}$ be the observed and $n\hat p_j$ the fitted count of cell $j$. The **Pearson statistic** is

$$
X^2=\sum_{j=1}^c\frac{(N_j-n\hat p_j)^2}{n\hat p_j} .
$$

**Theorem 8.5** (Pearson's statistic with known cell probabilities). Let $y_1,\ldots,y_n$ be independent and identically distributed with $p_j=\operatorname{\mathsf{P}}(y_1\in C_j)>0$ for every $j$, and take $\hat p_j=p_j$. Then $X^2\xrightarrow{\,d\,}\chi^2_{c-1}$.

**Proof.** Put $\mathbf{z}_i=(\mathbb{1}\{y_i\in C_1\}-p_1,\ldots,\mathbb{1}\{y_i\in C_{c-1}\}-p_{c-1})'$, the first $c-1$ cells only. The $\mathbf{z}_i$ are independent and identically distributed, bounded by $1$ in each coordinate, with mean $\mathbf{0}$ and covariance $\boldsymbol{\Sigma}=\mathbf{D}-\mathbf{p}\mathbf{p}'$, where $\mathbf{p}=(p_1,\ldots,p_{c-1})'$ and $\mathbf{D}=\operatorname{diag}(\mathbf{p})$, since $\mathbb{1}\{y\in C_j\}\mathbb{1}\{y\in C_l\}=\mathbb{1}\{y\in C_j\}\delta_{jl}$. By the prerequisites' lemma **Sherman–Morrison**, with $\mathbf 1$ the vector of ones and $1-\mathbf{p}'\mathbf{D}^{-1}\mathbf{p}=1-\sum_{j<c}p_j=p_c>0$,

$$
\boldsymbol{\Sigma}^{-1}=\mathbf{D}^{-1}+\frac{\mathbf{D}^{-1}\mathbf{p}\mathbf{p}'\mathbf{D}^{-1}}{1-\mathbf{p}'\mathbf{D}^{-1}\mathbf{p}}
=\mathbf{D}^{-1}+\frac{\mathbf 1\mathbf 1'}{p_c} ,
$$

so $\boldsymbol{\Sigma}$ is positive definite. The Lyapunov corollary of the prerequisites, with $\delta=1$, gives $\mathbf{V}_n=n^{-1/2}\sum_i\mathbf{z}_i\xrightarrow{\,d\,} N(\mathbf{0},\boldsymbol{\Sigma})$. Coordinate $j$ of $\mathbf{V}_n$ is $(N_j-np_j)/\sqrt n$, and $\sum_{j\leqslant c}(N_j-np_j)=n-n=0$, so $\mathbf 1'\mathbf{V}_n=-(N_c-np_c)/\sqrt n$. Hence

$$
\mathbf{V}_n'\boldsymbol{\Sigma}^{-1}\mathbf{V}_n=\sum_{j<c}\frac{(N_j-np_j)^2}{np_j}+\frac{(N_c-np_c)^2}{np_c}=X^2
$$

exactly, and the prerequisites' theorem **quadratic forms of asymptotically normal vectors**, with $\hat{\boldsymbol{\Sigma}}_n=\boldsymbol{\Sigma}$, gives the $\chi^2_{c-1}$ limit. ∎

**Lemma 8.6** (Cell probabilities of the Poisson). For $y\sim\mathrm{Pois}(\mu)$ and cells $C_1,\ldots,C_c$, let $p_j(\mu)=\operatorname{\mathsf{P}}(y\in C_j)$ and $m_j(\mu)=\operatorname{\mathsf{E}}[y\mid y\in C_j]$ when $p_j>0$. Then each $p_j$ is differentiable on $(0,\infty)$ with continuous derivative

$$
p_j'(\mu)=\frac{\operatorname{\mathsf{E}}[\mathbb{1}\{y\in C_j\}(y-\mu)]}{\mu}=\frac{p_j(m_j-\mu)}{\mu} ,
\qquad \sum_{j=1}^cp_j'(\mu)=0 ,
$$

and

$$
\kappa(\mu)=\mu\sum_{j=1}^c\frac{p_j'(\mu)^2}{p_j(\mu)}
=\frac{\sum_jp_j(m_j-\mu)^2}{\mu}\in[0,1) .
$$

**Proof.** $p_j(\mu)=\sum_{y\in C_j}g(y,\mu)$ with $g(y,\mu)=e^{-\mu}\mu^y/y!$, a sum against counting measure, and $\partial g/\partial\mu=g\cdot(y/\mu-1)$. For $\mu$ in $[\mu_0/2,2\mu_0]$, $|\partial g/\partial\mu|\leqslant e^{-\mu_0/2}(2\mu_0)^y(2y/\mu_0+1)/y!$, which is summable over $y$; the prerequisites' theorem **differentiation under the integral sign** gives $p_j'(\mu)=\sum_{y\in C_j}g(y,\mu)(y-\mu)/\mu=\operatorname{\mathsf{E}}[\mathbb{1}\{y\in C_j\}(y-\mu)]/\mu$. The same argument applied to this series, whose terms are $g$ times a polynomial in $y$ and $1/\mu$, shows $p_j'$ is itself differentiable, hence continuous. Summing over $j$ gives $\operatorname{\mathsf{E}}(y-\mu)/\mu=0$. Since $\operatorname{\mathsf{E}}[\mathbb{1}\{y\in C_j\}y]=p_jm_j$, the numerator is $p_j(m_j-\mu)$, and substituting gives the second form of $\kappa$. Finally, with $\mathbb{1}_j=\mathbb{1}\{y\in C_j\}$, $(y-\mu)^2=\sum_j\mathbb{1}_j\{(y-m_j)+(m_j-\mu)\}^2$ and $\operatorname{\mathsf{E}}[\mathbb{1}_j(y-m_j)]=0$, so

$$
\mu=\operatorname{var} y=\sum_j\operatorname{\mathsf{E}}[\mathbb{1}_j(y-m_j)^2]+\sum_jp_j(m_j-\mu)^2 .
$$

The first sum is positive: finitely many cells cover the infinitely many points of positive probability, so some cell contains two such points and $y-m_j$ is not almost surely zero on it. Hence $\kappa<1$. ∎

**Theorem 8.7** (Pearson's statistic with $\mu$ estimated by $\bar y$). Let $y_1,\ldots,y_n$ be independent $\mathrm{Pois}(\mu)$, let every $p_j(\mu)>0$, and take $\hat p_j=p_j(\bar y)$. Then, with $\kappa=\kappa(\mu)$ of Lemma 8.6 and $Z_1,\ldots,Z_{c-1}$ independent standard normal,

$$
X^2\xrightarrow{\,d\,}(1-\kappa)Z_1^2+\sum_{k=2}^{c-1}Z_k^2 .
$$

For every $x$ the limit $V$ satisfies $\operatorname{\mathsf{P}}(\chi^2_{c-2}>x)\leqslant\operatorname{\mathsf{P}}(V>x)\leqslant\operatorname{\mathsf{P}}(\chi^2_{c-1}>x)$, $\chi^2_0$ being the point mass at $0$.

**Proof.** Write $p_j=p_j(\mu)$, $a_j=p_j'(\mu)$ and $T_n=n^{-1/2}\sum_i(y_i-\mu)$.

**Step 1: linearising the fitted probabilities.** $\operatorname{\mathsf{E}} T_n^2=\mu$, so $T_n$ is bounded in probability by Chebyshev's inequality and $\bar y\xrightarrow{\,p\,}\mu$. On $\{\bar y>0\}$, of probability $1-e^{-n\mu}$, the mean value theorem gives $m_{nj}$ between $\bar y$ and $\mu$ with

$$
\sqrt n\{p_j(\bar y)-p_j\}=p_j'(m_{nj})T_n=a_jT_n+\{p_j'(m_{nj})-a_j\}T_n ,
$$

and the last term tends to $0$ in probability: $m_{nj}\xrightarrow{\,p\,}\mu$, $p_j'$ is continuous (Lemma 8.6), $T_n\xrightarrow{\,d\,} N(0,\mu)$ by the Lindeberg–Lévy corollary, and the continuous mapping theorem and Slutsky's theorem of the prerequisites apply.

**Step 2: a sum of independent vectors.** Let $W_{nj}=(N_j-np_j(\bar y))/\sqrt{np_j}$. By Step 1, $W_{nj}=n^{-1/2}\sum_i\xi_{ij}+o_p(1)$ with

$$
\xi_{ij}=\frac{\mathbb{1}\{y_i\in C_j\}-p_j-a_j(y_i-\mu)}{\sqrt{p_j}} .
$$

The vectors $\boldsymbol{\xi}_i=(\xi_{i1},\ldots,\xi_{ic})'$ are independent and identically distributed with mean $\mathbf{0}$. As $\operatorname{\mathsf{E}}[(\mathbb{1}\{y\in C_j\}-p_j)(y-\mu)]=\mu a_j$ by Lemma 8.6 and $\operatorname{var} y=\mu$,

$$
\operatorname{cov}(\xi_{ij},\xi_{il})
=\frac{p_j\delta_{jl}-p_jp_l-2\mu a_ja_l+\mu a_ja_l}{\sqrt{p_jp_l}} ,
\qquad\text{so}\qquad
\operatorname{var}\boldsymbol{\xi}_i=\mathbf{I}_c-\mathbf{u}\mathbf{u}'-\mathbf{v}\mathbf{v}' ,
$$

with $u_j=\sqrt{p_j}$ and $v_j=\sqrt\mu\,a_j/\sqrt{p_j}$. Here $|\mathbf{u}|^2=\sum_jp_j=1$, $\mathbf{u}'\mathbf{v}=\sqrt\mu\sum_ja_j=0$ and $|\mathbf{v}|^2=\kappa$.

**Step 3: an orthonormal basis.** Let $\mathbf{q}_1=\mathbf{u}$, let $\mathbf{q}_2=\mathbf{v}/\sqrt\kappa$ if $\kappa>0$ and any unit vector orthogonal to $\mathbf{u}$ otherwise, and complete to an orthonormal basis $\mathbf{q}_1,\ldots,\mathbf{q}_c$ of $\mathbb{R}^c$. Then $\mathbf{q}_1'\boldsymbol{\xi}_i=\sum_j\{\mathbb{1}\{y_i\in C_j\}-p_j-a_j(y_i-\mu)\}=1-1-0=0$ identically, and $\boldsymbol{\zeta}_i=(\mathbf{q}_2'\boldsymbol{\xi}_i,\ldots,\mathbf{q}_c'\boldsymbol{\xi}_i)'$ has covariance $\operatorname{diag}(1-\kappa,1,\ldots,1)$, positive definite because $\kappa<1$. Each coordinate of $\boldsymbol{\xi}_i$ is at most $(1+|a_j|\,|y_i-\mu|)/\sqrt{p_j}$ in absolute value, so $\operatorname{\mathsf{E}}|\boldsymbol{\zeta}_i|^3<\infty$ by Proposition 8.2(ii), and the Lyapunov corollary gives $n^{-1/2}\sum_i\boldsymbol{\zeta}_i\xrightarrow{\,d\,} N(\mathbf{0},\operatorname{diag}(1-\kappa,1,\ldots,1))$.

**Step 4: the limit of $|\mathbf{W}_n|^2$.** Since $\mathbf{q}_1,\ldots,\mathbf{q}_c$ is an orthonormal basis of $\mathbb{R}^c$, $|\mathbf{W}_n|^2=\sum_k(\mathbf{q}_k'\mathbf{W}_n)^2$. By Steps 2 and 3, $\mathbf{q}_1'\mathbf{W}_n=o_p(1)$ and $(\mathbf{q}_k'\mathbf{W}_n)_{k\geqslant2}=n^{-1/2}\sum_i\boldsymbol{\zeta}_i+o_p(1)$, so Slutsky's theorem and the continuous mapping theorem give $|\mathbf{W}_n|^2\xrightarrow{\,d\,}(1-\kappa)Z_1^2+\sum_{k=2}^{c-1}Z_k^2$; the limiting coordinates are uncorrelated jointly normal, hence independent by the prerequisites' theorem **uncorrelated blocks are independent**.

**Step 5: the denominators.** $X^2=\sum_jW_{nj}^2\,p_j/p_j(\bar y)$, and $p_j/p_j(\bar y)\xrightarrow{\,p\,}1$ by continuity, so $X^2-|\mathbf{W}_n|^2\xrightarrow{\,p\,}0$ and Slutsky's theorem finishes the proof. The bounds hold because the limit is $\sum_{k\geqslant2}Z_k^2$ plus the non-negative $(1-\kappa)Z_1^2$, and at most $\sum_{k\geqslant1}Z_k^2$. ∎

**Example 8.2** (Singapore, continued; FREES §12.1.1, exercise 12.2). FREES's last cell is $\{4\}$, and its five cells miss $\{5,6,\ldots\}$; the cells of Theorem 8.7 are $\{0\},\{1\},\{2\},\{3\},\{4,5,\ldots\}$:

In [ ]:
o  <- c(table(factor(pmin(y, 4), levels = 0:4)))
eF <- n * dpois(0:4, ybar)                                  # FREES's cells
eT <- n * c(dpois(0:3, ybar), ppois(3, ybar, lower.tail = FALSE))
X2 <- function(o, e) sum((o - e)^2 / e)
round(c(X2_frees = X2(o, eF), X2_tail_cell = X2(o, eT),
        cell3_term = (o[[4]] - eT[4])^2 / eT[4]), 3)
a <- c(dpois(0:3, ybar) * (0:3 / ybar - 1), dpois(3, ybar))  # p_j'(mu)
kap <- ybar * sum(a^2 / (eT / n))
signif(c(one_minus_kappa = 1 - kap, p_chisq2 = pchisq(X2(o, eT), 2, lower.tail = FALSE),
         p_chisq3 = pchisq(X2(o, eT), 3, lower.tail = FALSE)), 4)

FREES's $41.98$ is reproduced, and the tail cell changes it in the fourth decimal place. The cell $\{3\}$, with fitted count $0.40$, contributes $32.7$ of it. Here $1-\kappa=1.9\times10^{-7}$, so the limit of Theorem 8.7 is $\chi^2_3$ up to a summand $1.9\times10^{-7}Z_1^2$, and FREES's $\operatorname{df}=5-1-1=3$ gives a $p$-value of order $10^{-9}$. That limit fixes the cells and lets $n\to\infty$; at $n=7483$ the fitted count of cell $\{3\}$ is $0.40$. The null law at the actual $n$, by simulation:

In [ ]:
set.seed(812)
Xsim <- replicate(20000, {
  ys <- rpois(n, ybar); m <- mean(ys)
  os <- c(table(factor(pmin(ys, 4), levels = 0:4)))
  X2(os, n * c(dpois(0:3, m), ppois(3, m, lower.tail = FALSE))) })
c(p_simulated = mean(Xsim >= X2(o, eT)),
  quantile95_simulated = unname(quantile(Xsim, 0.95)), quantile95_chisq3 = qchisq(0.95, 3))

The simulated $p$-value is $0.007$, six orders of magnitude above the $\chi^2_3$ value: four counts of $3$ in a cell expecting $0.40$ are rare, but not as rare as the limit says. The Poisson is still rejected.

### 1.3 Linear regression on counts

**Example 8.3** (Bike sharing; ISLR §4.6.1, Table 4.10). ISLR's `Bikeshare`: $n=8645$ hours of a Washington, DC bike-sharing programme, with `bikers` the hourly count of users, regressed by least squares on `mnth` and `hr` (factors coded to sum to zero, as in ISLR), `workingday`, `temp` and `weathersit`:

In [ ]:
B <- Bikeshare
contrasts(B$hr) <- contr.sum(24); contrasts(B$mnth) <- contr.sum(12)
fB <- bikers ~ mnth + hr + workingday + temp + weathersit
mlm <- lm(fB, data = B)
round(coef(summary(mlm))[c(1, 36:40), ], 2)
c(min_bikers = min(B$bikers), negative_fitted = mean(fitted(mlm) < 0))
hr <- as.numeric(as.character(B$hr))
night <- B$bikers[hr %in% 1:3 & B$mnth %in% c("Dec", "Jan", "Feb") &
                  B$weathersit == "light rain/snow"]
morning <- B$bikers[hr %in% 7:9 & B$mnth %in% c("April", "May", "June") &
                    B$weathersit == "clear"]
round(c(mean_night = mean(night), sd_night = sd(night),
        mean_morning = mean(morning), sd_morning = sd(morning)), 2)

ISLR's Table 4.10 is reproduced, two entries differing in the last printed digit ($-22.42$ and $-109.74$ here, $-22.43$ and $-109.75$ in ISLR). $9.6\%$ of the fitted values are negative. ISLR's two subsets, “between 1 AM and 4 AM” and “between 7 AM and 10 AM”, are the hours $1$–$3$ and $7$–$9$: means $5.05$ and $243.59$, standard deviations $3.73$ and $131.70$.

**Proposition 8.8** (Three drawbacks of a linear mean for counts; ISLR §4.6.1). Let $y_1,\ldots,y_n$ be counts with $\operatorname{\mathsf{E}} y_i=\mathbf{x}_i'\boldsymbol{\beta}$.

(i) If $\mathbf{x}'\boldsymbol{\beta}\geqslant0$ for every $\mathbf{x}=(1,x_1,\ldots,x_k)'$ with $x_1,\ldots,x_k\in\mathbb{R}$, then $\beta_1=\cdots=\beta_k=0$.

(ii) If the $y_i$ are Poisson, $\operatorname{var} y_i=\mathbf{x}_i'\boldsymbol{\beta}$, and A3 holds only if all the means are equal.

(iii) $y_i-\mathbf{x}_i'\boldsymbol{\beta}$ takes only the values $j-\mathbf{x}_i'\boldsymbol{\beta}$, $j=0,1,\ldots$, so A5 fails.

**Proof.** (i) If $\beta_j\neq0$, the vector with $x_j=t$ and the other slopes' coordinates $0$ gives $\beta_0+t\beta_j$, negative for $t$ of the sign opposite to $\beta_j$ and large. (ii) Proposition 8.2(ii). (iii) A normal variable gives positive probability to every interval, and $y_i-\mathbf{x}_i'\boldsymbol{\beta}$ gives probability $0$ to the open interval $(-\mathbf{x}_i'\boldsymbol{\beta},1-\mathbf{x}_i'\boldsymbol{\beta})$ between two of its consecutive values. ∎

**Remark** (ISLR §4.6.1). Regressing $\ln y_i$ instead removes negative predictions and much of the heteroscedasticity, but is undefined at $y_i=0$ and models $\operatorname{\mathsf{E}}\ln y$, not $\ln\operatorname{\mathsf{E}} y$. `Bikeshare` has no zero count, the smallest being $1$.

## 2 Poisson Regression

> **Reading.** FREES §12.1.2–12.1.4; ISLR §4.6.2 (PA reading list, not SRM). SRM learning outcomes 2(b), 2(c), 2(e), 2(f).

### 2.1 The model, exposure and the offset

**Definition 8.9** (Poisson regression; FREES (12.4), ISLR (4.36)). Given fixed $\mathbf{x}_i=(1,x_{i1},\ldots,x_{ik})'$, the rows of $\mathbf{X}$, and known **exposures** $E_i>0$, the **Poisson regression model** takes $y_1,\ldots,y_n$ independent with

$$
y_i\sim\mathrm{Pois}(\mu_i) , \qquad \ln\mu_i=\ln E_i+\mathbf{x}_i'\boldsymbol{\beta} . \tag{3}
$$

The function $\ln$ connecting $\mu_i$ to $\mathbf{x}_i'\boldsymbol{\beta}$ is the **logarithmic link**; the term $\ln E_i$, a covariate whose coefficient is fixed at $1$, is an **offset**. Without exposures, $E_i=1$.

**Proposition 8.10** (Interpretation; FREES §12.1.2, ISLR §4.6.2). Under (3):

(i) $\mu_i=E_ie^{\mathbf{x}_i'\boldsymbol{\beta}}>0$ for every $\boldsymbol{\beta}$;

(ii) raising $x_{ij}$ by one unit, the other coordinates and $E_i$ fixed, multiplies $\mu_i$ by $e^{\beta_j}$, and $\beta_j=\partial\ln\mu_i/\partial x_{ij}$ is the proportional change in the mean per unit change of $x_{ij}$;

(iii) the mean is proportional to the exposure;

(iv) with indicator covariates for the levels of two factors, $\mu_i=E_i\cdot B\cdot A_{a(i)}\cdot T_{u(i)}$ with $B=e^{\beta_0}$ and one factor per level: a **multiplicative** structure.

**Proof.** (i) Exponentiate. (ii) The ratio of the two means is $e^{(x_{ij}+1)\beta_j-x_{ij}\beta_j}=e^{\beta_j}$; differentiate $\ln\mu_i$. (iii) $\mu_i/E_i$ does not involve $E_i$. (iv) With $\mathbf{x}_i'\boldsymbol{\beta}=\beta_0+\alpha_{a(i)}+\tau_{u(i)}$, exponentiate term by term. ∎

**Remark** (Exposure; FREES §12.1.2). By Proposition 8.2(iii) a policy observed for $m$ periods, with independent $\mathrm{Pois}(\mu)$ counts in each, has a $\mathrm{Pois}(m\mu)$ total: proportionality to exposure is a theorem for integer exposures and an assumption for fractional ones, such as the fraction of the year a Singapore policy was in force. Entering $\ln E_i$ as an ordinary covariate with coefficient $\gamma$ fits $\mu_i=E_i^\gamma e^{\mathbf{x}_i'\boldsymbol{\beta}}$, which drops (iii) unless $\gamma=1$. Weber's California study (FREES §12.1.2) used the linear mean $\operatorname{\mathsf{E}} y_i=\mathbf{x}_i'\boldsymbol{\beta}$, which Proposition 8.8(i) restricts.

### 2.2 Estimation

**Theorem 8.11** (Score, information, concavity; FREES §12.1.3). Under (3) the log-likelihood, score and information are

$$
\begin{align}
L(\boldsymbol{\beta})&=\sum_{i=1}^n\bigl\{-E_ie^{\mathbf{x}_i'\boldsymbol{\beta}}+y_i(\ln E_i+\mathbf{x}_i'\boldsymbol{\beta})-\ln y_i!\bigr\} , \tag{4} \\
\nabla L(\boldsymbol{\beta})&=\sum_{i=1}^n(y_i-\mu_i)\mathbf{x}_i=\mathbf{X}'(\mathbf{Y}-\boldsymbol{\mu}) ,
\qquad
\mathcal{I}(\boldsymbol{\beta})=\sum_{i=1}^n\mu_i\mathbf{x}_i\mathbf{x}_i'=\mathbf{X}'\mathbf{W}\mathbf{X} , \tag{5}
\end{align}
$$

with $\mathbf{W}=\operatorname{diag}(\mu_i)$, and $-\nabla^2L(\boldsymbol{\beta})=\mathcal{I}(\boldsymbol{\beta})$ whatever the responses. $L$ is concave, and strictly concave when $\operatorname{rank}(\mathbf{X})=k+1$.

**Proof.** Sum the logarithm of (1) at $\mu_i=E_ie^{\mathbf{x}_i'\boldsymbol{\beta}}$. As $\partial\mu_i/\partial\boldsymbol{\beta}=\mu_i\mathbf{x}_i$, $\nabla L=\sum_i(y_i-\mu_i)\mathbf{x}_i$ and $\nabla^2L=-\sum_i\mu_i\mathbf{x}_i\mathbf{x}_i'$, free of the $y_i$. The score has mean $\mathbf{0}$ and independent summands of variance $\mu_i\mathbf{x}_i\mathbf{x}_i'$, so its variance is $\sum_i\mu_i\mathbf{x}_i\mathbf{x}_i'$. For $\mathbf{c}\neq\mathbf{0}$, $\mathbf{c}'\mathcal{I}\mathbf{c}=\sum_i\mu_i(\mathbf{x}_i'\mathbf{c})^2$ is non-negative, and positive unless $\mathbf{X}\mathbf{c}=\mathbf{0}$; the prerequisites' lemma **convexity from the derivative**, applied along lines, gives the concavity and the strict concavity. ∎

**Corollary 8.12** (The score equations; FREES §12.1.3). A maximiser $\mathbf b$, with $\hat\mu_i=E_ie^{\mathbf{x}_i'\mathbf b}$, satisfies $\sum_ix_{ij}\hat\mu_i=\sum_ix_{ij}y_i$ for $j=0,\ldots,k$. With the intercept, $\sum_i(y_i-\hat\mu_i)=0$, and the fitted and observed counts agree in total and within every level of a factor entered by indicators.

**Proof.** At an interior maximiser of the differentiable $L$ the gradient $\mathbf{X}'(\mathbf{Y}-\hat{\boldsymbol{\mu}})$ vanishes; read it coordinate by coordinate, with $x_{i0}=1$ and $x_{ij}$ an indicator. ∎

**Theorem 8.13** (Existence). Let $\operatorname{rank}(\mathbf{X})=k+1$.

(i) If some $\mathbf{c}\neq\mathbf{0}$ has $\mathbf{x}_i'\mathbf{c}\leqslant0$ for every $i$ and $\mathbf{x}_i'\mathbf{c}=0$ whenever $y_i>0$, no maximum likelihood estimate exists.

(ii) Otherwise $L$ has exactly one maximiser.

**Proof.** Write $-L(\boldsymbol{\beta})=\sum_ih_i(\mathbf{x}_i'\boldsymbol{\beta})+\mathrm{const}$ with $h_i(\eta)=E_ie^\eta-y_i\eta$, convex. If $y_i>0$, $h_i$ decreases on $(-\infty,s_i]$ and increases on $[s_i,\infty)$, where $s_i=\ln(y_i/E_i)$; if $y_i=0$, $h_i$ increases on $\mathbb{R}$.

(i) For any $\boldsymbol{\beta}$, $L(\boldsymbol{\beta}+t\mathbf{c})=\sum_i\{y_i(\ln E_i+\mathbf{x}_i'\boldsymbol{\beta})-E_ie^{\mathbf{x}_i'\boldsymbol{\beta}} e^{t\mathbf{x}_i'\mathbf{c}}-\ln y_i!\}$, as $y_i\mathbf{x}_i'\mathbf{c}=0$. Some $\mathbf{x}_i'\mathbf{c}<0$, since otherwise $\mathbf{X}\mathbf{c}=\mathbf{0}$; so $L(\boldsymbol{\beta}+t\mathbf{c})$ is strictly increasing in $t$ and no $\boldsymbol{\beta}$ maximises $L$.

(ii) For a unit vector $\mathbf{c}$ put $m(\mathbf{c})=\max\{\max_i\mathbf{x}_i'\mathbf{c},\ \max_{i:y_i>0}(-\mathbf{x}_i'\mathbf{c})\}$. If $m(\mathbf{c})\leqslant0$, every $\mathbf{x}_i'\mathbf{c}\leqslant0$ and $\mathbf{x}_i'\mathbf{c}=0$ whenever $y_i>0$, excluded; so $m>0$ on the unit sphere. $m$ is continuous, the sphere is compact by the prerequisites' theorem **Heine–Borel**, and $m$ attains a minimum $m_0>0$ by the theorem **uniform continuity on compacts**. Let $h_i^*=\inf h_i$, which is $0$ if $y_i=0$ and $h_i(s_i)$ if $y_i>0$, and let $t>\max_{i:y_i>0}|s_i|/m_0$. If $m(\mathbf{c})=\mathbf{x}_j'\mathbf{c}$, then $t\mathbf{x}_j'\mathbf{c}\geqslant tm_0>s_j$ and $h_j(t\mathbf{x}_j'\mathbf{c})\geqslant h_j(tm_0)$; if $m(\mathbf{c})=-\mathbf{x}_j'\mathbf{c}$ with $y_j>0$, then $t\mathbf{x}_j'\mathbf{c}\leqslant-tm_0<s_j$ and $h_j(t\mathbf{x}_j'\mathbf{c})\geqslant h_j(-tm_0)$. Bounding every other $h_i$ by $h_i^*$,

$$
-L(t\mathbf{c})-\mathrm{const}\geqslant\sum_ih_i^*+\min_j\bigl\{g_j(t)-h_j^*\bigr\} ,
$$

where $g_j(t)=\min\{h_j(tm_0),h_j(-tm_0)\}$ if $y_j>0$ and $g_j(t)=h_j(tm_0)$ if $y_j=0$. Each $g_j(t)\to\infty$, since $h_j(tm_0)\geqslant E_je^{tm_0}-y_jtm_0$ and $h_j(-tm_0)\geqslant y_jtm_0$, and the minimum is over finitely many $j$, so $-L(t\mathbf{c})\to\infty$ uniformly in $\mathbf{c}$. Hence $-L$ is continuous and coercive and has a minimiser by the prerequisites' lemma **coercive minimisation**, unique by strict convexity (Theorem 8.11). ∎

**Corollary 8.14** (An empty level). If a factor is entered by indicators against a reference level and some non-reference level, observed at least once, has $y_i=0$ for every observation in it, no maximum likelihood estimate exists.

**Proof.** Take $\mathbf{c}=-\mathbf e_l$ for that level's indicator $l$: $\mathbf{x}_i'\mathbf{c}=-1$ on the level and $0$ elsewhere, and Theorem 8.13(i) applies. ∎

### 2.3 Large-sample inference

**Theorem 8.15** (Sampling distribution; FREES §12.1.3). Let (3) hold with true parameter $\boldsymbol{\beta}_0$, and suppose

(D1) $|\mathbf{x}_i|\leqslant M$ and $c_E\leqslant E_i\leqslant C_E$ for constants $M,c_E,C_E>0$ free of $i$;

(D2) $\mathbf{F}_n=n^{-1}\sum_{i\leqslant n}\mu_i\mathbf{x}_i\mathbf{x}_i'\to\mathbf{F}$ positive definite, the $\mu_i$ evaluated at $\boldsymbol{\beta}_0$.

Then with probability tending to one the maximum likelihood estimate $\mathbf b$ exists and is unique, $\mathbf b\xrightarrow{\,p\,}\boldsymbol{\beta}_0$,

$$
\sqrt n(\mathbf b-\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{F}^{-1}) ,
\qquad n^{-1}\mathcal{I}(\mathbf b)\xrightarrow{\,p\,}\mathbf{F} ,
$$

and the Wald and likelihood ratio statistics for $q$ linear restrictions $\mathbf{R}\boldsymbol{\beta}=\mathbf{r}$ that hold at $\boldsymbol{\beta}_0$ are $\chi^2_q$ in the limit.

**Proof.** The hypotheses of the prerequisites' theorem **concave maximum likelihood** are checked with $\Theta=\mathbb{R}^{k+1}$ and $\ell_i$ the $i$th summand of (4).

**Concavity and smoothness.** Theorem 8.11.

**The score condition.** $\nabla\ell_i(\boldsymbol{\beta}_0)=(y_i-\mu_i)\mathbf{x}_i$ has mean $\mathbf{0}$ and variance $\mu_i\mathbf{x}_i\mathbf{x}_i'$, whose average is $\mathbf{F}_n\to\mathbf{F}$. Under (D1) every $\mu_i=E_ie^{\mathbf{x}_i'\boldsymbol{\beta}_0}$ lies in $[m_-,m_+]$ with $m_-=c_Ee^{-M|\boldsymbol{\beta}_0|}>0$ and $m_+=C_Ee^{M|\boldsymbol{\beta}_0|}$, so by Proposition 8.2(ii)

$$
\operatorname{\mathsf{E}}|(y_i-\mu_i)\mathbf{x}_i|^4\leqslant M^4(m_++3m_+^2) ,
$$

and the Lyapunov corollary with $\delta=2$ gives $n^{-1/2}\nabla L(\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{F})$. The summands are unbounded, so the bounded-summand corollary does not apply.

**The curvature condition.** $\mathbf{H}_n(\boldsymbol{\beta})=\sum_iE_ie^{\mathbf{x}_i'\boldsymbol{\beta}}\mathbf{x}_i\mathbf{x}_i'$ is non-random and $n^{-1}\mathbf{H}_n(\boldsymbol{\beta}_0)=\mathbf{F}_n\to\mathbf{F}$, so $\mathbf{G}=\mathbf{F}$, the information equality. For $|\boldsymbol{\beta}-\boldsymbol{\beta}_0|\leqslant1$ the exponents lie in $[-M(|\boldsymbol{\beta}_0|+1),M(|\boldsymbol{\beta}_0|+1)]$, where $e^{\eta}\leqslant K=e^{M(|\boldsymbol{\beta}_0|+1)}$, so the mean value theorem gives $|e^{\mathbf{x}_i'\boldsymbol{\beta}}-e^{\mathbf{x}_i'\boldsymbol{\beta}_0}|\leqslant KM|\boldsymbol{\beta}-\boldsymbol{\beta}_0|$ and $\|\mathbf{H}_n(\boldsymbol{\beta})-\mathbf{H}_n(\boldsymbol{\beta}_0)\|\leqslant nC_EKM^3|\boldsymbol{\beta}-\boldsymbol{\beta}_0|$, a constant Lipschitz bound.

The conclusions are parts (i)–(v) of that theorem with $\mathbf{F}=\mathbf{G}$, unique because $L$ is strictly concave once $\operatorname{rank}\mathbf{X}=k+1$, which (D2) forces for large $n$ (two maximisers would give a strictly larger value at their midpoint); and its corollary **Wald** and theorem **likelihood ratio**, whose restricted problem satisfies (D1)–(D2) with the corresponding sub-block of $\mathbf{F}$. For restrictions $\mathbf{R}\boldsymbol{\beta}=\mathbf{r}$ with $\mathbf{R}$ of full row rank $q$, the likelihood ratio theorem is applied in the coordinates $\boldsymbol{\gamma}=\mathbf{A}^{-1}\boldsymbol{\beta}=(\mathbf{S}\boldsymbol{\beta},\mathbf{R}\boldsymbol{\beta})$, the rows of $\mathbf{S}$ completing those of $\mathbf{R}$ to a basis: the covariates $\mathbf{A}'\mathbf{x}_i$ keep (D1)–(D2) with information $\mathbf{A}'\mathbf{F}\mathbf{A}$, the null is the coordinate restriction $\boldsymbol{\gamma}_2=\mathbf{r}$, and the likelihood ratio is unchanged. ∎

**Remark** (FREES §12.1.3–12.1.4). The standard error of $b_j$ is the square root of the $j$th diagonal entry of $\mathcal{I}(\mathbf b)^{-1}=(\sum_i\hat\mu_i\mathbf{x}_i\mathbf{x}_i')^{-1}$, and $b_j/\operatorname{se}(b_j)$ is the $t$-ratio (R's “$z$ value”). FREES's $\text{AIC}=-2L(\mathbf b)+2(k+1)$ and its likelihood ratio statistic $\text{LRT}=2\{L(\mathbf b)-L(\mathbf b_{\mathrm{Reduced}})\}$ are those of Unit 7.

**Remark** (Medical malpractice; FREES §12.1.3, Table 12.2). Fournier and McInnes fitted a Poisson regression to the malpractice claims of $n=9059$ Florida physicians, 1985–1989; FREES prints their coefficients, not their data. The coefficient of the female indicator is $-0.432$ with standard error $0.082$, so a male physician has $e^{0.432}=1.540$ times the expected number of claims of a female one with the same other characteristics (Proposition 8.10), and by Theorem 8.15 with the standard error just defined (the prerequisites' corollary **Wald** with $q=1$) an approximate $95\%$ interval for that factor is $\exp(0.432\pm1.96\times0.082)=(1.31,1.81)$. Speciality matters most: surgery's $1.410$ multiplies the expected count by $e^{1.410}=4.10$ relative to the reference speciality.

**Example 8.4** (Bike sharing, Poisson; ISLR §4.6.2, Table 4.11).

In [ ]:
mpo <- glm(fB, data = B, family = poisson)
round(coef(summary(mpo))[c(1, 36:40), ], 2)
cw <- coef(mpo)[c("weathersitcloudy/misty", "weathersitlight rain/snow")]
round(c(cloudy_vs_clear = exp(cw[[1]]), rain_vs_cloudy = exp(cw[[2]] - cw[[1]]),
        from_rounded = exp(c(-0.08, -0.50))), 4)

ISLR's Table 4.11 is reproduced, one $z$ value differing in the last digit ($-141.90$ against $-141.91$). By Proposition 8.10(ii) cloudy skies multiply the mean by $e^{-0.07523}=0.9275$ and rain multiplies it by a further $0.6062$; ISLR's $0.923$ and $0.607$ exponentiate the rounded $-0.08$ and $-0.50$. `workingday` has $z=7.50$ here and $t=0.71$ in Example 8.3.

### 2.4 Pearson residuals and the Pearson statistic

**Definition 8.16** (Pearson residuals; FREES §12.1.4). The **Pearson residuals** and the observation-level **Pearson statistic** of a fitted Poisson regression with $k+1$ coefficients are

$$
r_i=\frac{y_i-\hat\mu_i}{\sqrt{\hat\mu_i}} , \qquad
X^2=\sum_{i=1}^nr_i^2 , \qquad \hat\phi=\frac{X^2}{n-(k+1)} .
$$

**Proposition 8.17** (What the Pearson statistic estimates; FREES §12.1.4). Let $y_1,\ldots,y_n$ be independent with $\operatorname{\mathsf{E}} y_i=\mu_i=E_ie^{\mathbf{x}_i'\boldsymbol{\beta}_0}$ and $\operatorname{var} y_i=\phi\mu_i$, let (D1) hold, let $\sup_i\operatorname{\mathsf{E}}(y_i-\mu_i)^4<\infty$, and let $\mathbf b-\boldsymbol{\beta}_0$ be bounded in probability after multiplication by $\sqrt n$. Then $(y_i-\mu_i)/\sqrt{\mu_i}$ has variance exactly $\phi$, and $\hat\phi\xrightarrow{\,p\,}\phi$. Under (3), $\phi=1$ and $X^2/\{n-(k+1)\}\xrightarrow{\,p\,}1$.

**Proof.** The variance statement is $\operatorname{var} y_i/\mu_i=\phi$. Write $e_i=y_i-\mu_i$ and $\delta_i=\hat\mu_i-\mu_i$, and let $m_->0$ and $m_+$ bound the $\mu_i$ below and above (proof of Theorem 8.15). On $|\mathbf b-\boldsymbol{\beta}_0|\leqslant1$ the mean value theorem gives $\Delta=\max_i|\delta_i|\leqslant C_EKM|\mathbf b-\boldsymbol{\beta}_0|$, so $\Delta\to0$ in probability.

The average $\tilde\phi=n^{-1}\sum_ie_i^2/\mu_i$ has mean $\phi$ and variance at most $\sup_i\operatorname{\mathsf{E}} e_i^4/(nm_-^2)$, so $\tilde\phi\xrightarrow{\,p\,}\phi$ by Chebyshev's inequality. On $\{\Delta\leqslant m_-/2\}$, every $\hat\mu_i\geqslant m_-/2$ and

$$
\frac{(e_i-\delta_i)^2}{\hat\mu_i}-\frac{e_i^2}{\mu_i}
=-\frac{e_i^2\delta_i}{\mu_i\hat\mu_i}+\frac{\delta_i^2-2e_i\delta_i}{\hat\mu_i} ,
$$

so the average of these differences is at most $2\Delta m_-^{-2}n^{-1}\sum_ie_i^2+2m_-^{-1}(\Delta^2+2\Delta n^{-1}\sum_i|e_i|)$ in absolute value. Here $n^{-1}\sum_ie_i^2\leqslant m_+\tilde\phi$ and $n^{-1}\sum_i|e_i|\leqslant(m_+\tilde\phi)^{1/2}$ by the Cauchy–Schwarz inequality, so the bound is at most a continuous function of $(\Delta,\tilde\phi)$ that vanishes at $(0,\phi)$; as $(\Delta,\tilde\phi)\xrightarrow{\,p\,}(0,\phi)$, the continuous mapping theorem sends it to $0$ in probability. With $n/\{n-(k+1)\}\to1$, $\hat\phi\xrightarrow{\,p\,}\phi$. ∎

**Remark** (FREES §12.1.4). Plots of $r_i$ against fitted values or a candidate covariate locate unusual observations and omitted structure, and $X^2$ is a goodness-of-fit statistic. FREES's “$X^2$ should be approximately $n-(k+1)$” is Proposition 8.17: the ratio tends to $1$.

## 3 Application: Singapore Automobile Insurance

> **Reading.** FREES §12.2. SRM learning outcomes 2(c), 2(d), 2(e).

**Example 8.5** (Vehicle and personal characteristics; FREES Tables 12.4–12.5). `VehicleType` `A` is an automobile; `VAgecat1` groups vehicle age as $0$–$2$, $3$–$5$, $6$–$10$, $11$–$15$, $16+$ (codes $2$–$6$); `AgeCat` codes $2$–$7$ are the ages $22$–$25$, $26$–$35$, $36$–$45$, $46$–$55$, $56$–$65$ and $66+$; `NCD` is the no-claims discount in percent.

In [ ]:
S$Auto <- 1 * (S$VehicleType == "A")
table(Auto = S$Auto, count = y)
table(VAgecat1 = S$VAgecat1, count = y)
SA <- subset(S, Auto == 1)
rbind(n = table(SA$SexInsured), zero = table(SA$SexInsured[SA$Clm_Count == 0]))
round(100 * prop.table(table(S$VAgecat1, y == 0), 1)[, 2], 1); sum(S$Auto == 0)
round(100 * prop.table(table(SA$Clm_Count == 0, SA$AgeCat), 2)[2, ], 1)
round(100 * prop.table(table(SA$Clm_Count == 0, SA$NCD), 2)[2, ], 1)

FREES's Tables 12.4 and 12.5 are reproduced: $3842$ automobiles, of which $700$ have a female insured; the percentage with no claim is lowest ($91.8$) for vehicles aged $3$–$5$ and highest ($98.2$) for those aged $16$ or more, and rises from $89.6$ at `NCD` $=0$ to $94.8$ at $40$ ($94.4$ at $50$). Sex and age are recorded for automobiles only; the remaining $3641$ vehicles carry `SexInsured` `U` (three carry `M`) and `AgeCat` $0$ (one carries $4$), which the model uses only through $\texttt{Auto}=0$.

**Example 8.6** (The fitted Poisson model; FREES Table 12.6). FREES's model, from its chapter script (whose reference level `ref = "7"` for `AgeCatF` does not reproduce Table 12.6; `ref = "2"` does): `Female`, `Auto`, the age and discount factors interacted with `Auto`, vehicle age, and the offset $\ln E_i$ (`LNWEIGHT`), $E_i$ being the fraction of the year in force.

In [ ]:
S$NCD1F <- relevel(factor(S$NCD), ref = "50")
S$VAgecat1F <- relevel(factor(S$VAgecat1), ref = "6")
S$AgeCatF <- relevel(factor(S$AgeCat), ref = "2")
fS <- Clm_Count ~ Female + Auto + Auto:AgeCatF + Auto:NCD1F + VAgecat1F
m3 <- glm(fS, offset = LNWEIGHT, family = poisson, data = S)
round(coef(summary(m3))[, c(1, 3)], 3)
lfact <- sum(lfactorial(y))
print(c(logLik = as.numeric(logLik(m3)), sum_log_y_factorial = lfact,
        logLik_plus_sum = as.numeric(logLik(m3)) + lfact), digits = 7)

Every estimate and $t$-ratio of FREES's Table 12.6 is reproduced, with two corrections to the table's labels. The reference age level is $22$–$25$, not “$66$ and over” as its footnote says, and its five age rows are printed in reverse: FREES's “$22$–$25$” row ($0.747$, $t=0.961$) is the level $66+$, and its “$56$–$65$” row ($0.165$) is the level $26$–$35$. The level `AgeCat` $0$ occurs only with $\texttt{Auto}=0$, so `Auto:AgeCatF0` is identically zero and the indicator is dropped. FREES's $L(\mathbf b)=-1776.730$ omits the constant $-\sum_i\ln y_i!=-26.575$ of (4); R's $-1803.306$ includes it. The constant cancels in every likelihood ratio.

**Example 8.7** (Interpretation and a likelihood ratio test; FREES §12.2).

In [ ]:
b3 <- coef(m3)
round(c(NCD0_vs_50 = exp(b3[["Auto:NCD1F0"]]),
        NCD0_vs_30 = exp(b3[["Auto:NCD1F0"]] - b3[["Auto:NCD1F30"]]),
        frees_0_minus_20 = exp(b3[["Auto:NCD1F0"]] - b3[["Auto:NCD1F20"]])), 3)
m3r <- glm(Clm_Count ~ Auto:NCD1F + VAgecat1F, offset = LNWEIGHT,
           family = poisson, data = S)
LRT <- 2 * (as.numeric(logLik(m3)) - as.numeric(logLik(m3r)))
dfr <- df.residual(m3r) - df.residual(m3)
round(c(LRT = LRT, df = dfr, p_value = pchisq(LRT, dfr, lower.tail = FALSE),
        p_frees_df7 = pchisq(LRT, 7, lower.tail = FALSE)), 3)

An insured with `NCD` $=0$ has $e^{0.7293}=2.074$ times the expected claims of one with $50$, by Proposition 8.10(ii). Against `NCD` $=30$ the factor is $e^{0.7293-0.2597}=1.599$; FREES's $1.55$ subtracts $0.293$, the coefficient of `NCD` $=20$. FREES drops `Auto`, `Female` and the five age coefficients and refers $\text{LRT}=5.379$ to $\chi^2_7$. The reduced model that reproduces FREES's $L(\mathbf b_{\rm Reduced})=-1779.420$ (its script fits no reduced Poisson model) keeps `Auto:NCD1F`, which without the main effect has six columns, one per discount level, and their sum is `Auto`; so the reduced model is the full one with six restrictions, not seven, and by Theorem 8.15 the reference law is $\chi^2_6$: $p=0.496$, against $0.614$ from $\chi^2_7$ (FREES prints $0.618$).

**Example 8.8** (Comparing fitted frequency models; FREES Table 12.7). For a fitted model the fitted count of cell $j$ is $\sum_i\operatorname{\mathsf{P}}(y_i\in C_j;\hat\mu_i)$:

In [ ]:
cells <- function(P) { e <- colSums(P); c(round(e, 2), X2 = round(X2(o, e), 2)) }
Ppois <- function(mu) sapply(0:4, function(j) dpois(j, mu))
m1 <- glm(Clm_Count ~ 1, family = poisson, data = S)
m2 <- glm(Clm_Count ~ 1, offset = LNWEIGHT, family = poisson, data = S)
rbind(no_exposure = cells(Ppois(fitted(m1))), exposure = cells(Ppois(fitted(m2))),
      poisson = cells(Ppois(fitted(m3))))

The three Poisson columns of FREES's Table 12.7 and their statistics $41.98$, $17.62$ and $8.77$ are reproduced, with FREES's five cells. FREES refers $8.77$ to $\chi^2_4$, $p=0.067$, although its own rule, cells minus one minus estimated parameters, gives $5-1-17<0$. Neither Theorem 8.5 nor Theorem 8.7 covers seventeen coefficients estimated from the individual counts. The null law, by a parametric bootstrap from the fitted model:

In [ ]:
set.seed(1207); mu3 <- fitted(m3); Sb <- S
Xb <- replicate(400, {
  Sb$Clm_Count <- rpois(n, mu3); yb <- Sb$Clm_Count
  mb <- glm(fS, offset = LNWEIGHT, family = poisson, data = Sb)
  ob <- c(table(factor(pmin(yb, 4), levels = 0:4)))
  X2(ob, colSums(Ppois(fitted(mb)))) })
round(c(observed = X2(o, colSums(Ppois(mu3))), p_bootstrap = mean(Xb >= 8.7658),
        p_chisq4 = pchisq(8.7658, 4, lower.tail = FALSE)), 3)

The bootstrap $p$-value is $0.055$ against $\chi^2_4$'s $0.067$.

## 4 Overdispersion and the Negative Binomial

> **Reading.** FREES §12.3; ISLR §4.6.2 (footnote on overdispersion; PA reading list, not SRM). SRM learning outcomes 2(a), 2(d), 2(e).

### 4.1 Quasi-likelihood and robust standard errors

**Definition 8.18** (Over- and underdispersion; FREES §12.3). Counts with $\operatorname{var} y_i>\operatorname{\mathsf{E}} y_i$ are **overdispersed**, with $\operatorname{var} y_i<\operatorname{\mathsf{E}} y_i$ **underdispersed**, and with equality **equidispersed**.

**Theorem 8.19** (Quasi-Poisson; FREES (12.8)–(12.9)). Let $y_1,\ldots,y_n$ be independent with

$$
\operatorname{\mathsf{E}} y_i=\mu_i=E_ie^{\mathbf{x}_i'\boldsymbol{\beta}_0} , \qquad \operatorname{var} y_i=\phi\mu_i , \tag{6}
$$

$\phi>0$, $\sup_i\operatorname{\mathsf{E}}(y_i-\mu_i)^4<\infty$, and (D1), (D2) of Theorem 8.15. Let $\mathbf b$ maximise the Poisson log-likelihood (4), whatever the law of the $y_i$. Then

(i) $\mathbf b$ depends on the data only through the score equations $\mathbf{X}'(\mathbf{Y}-\boldsymbol{\mu})=\mathbf{0}$;

(ii) $\mathbf b\xrightarrow{\,p\,}\boldsymbol{\beta}_0$ and $\sqrt n(\mathbf b-\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\phi\mathbf{F}^{-1})$;

(iii) consistent standard errors are $\sqrt{\hat\phi}$ times the Poisson ones, $\hat\phi$ of Definition 8.16.

**Proof.** (i) is Corollary 8.12: nothing beyond (4) enters the computation.

(ii) Apply the prerequisites' theorem **concave maximum likelihood** with $\ell_n$ the Poisson log-likelihood, now a criterion rather than the log-likelihood of the data. Concavity and the curvature condition are those of Theorem 8.15, since $\mathbf{H}_n$ involves no $y_i$; so $\mathbf{G}=\mathbf{F}$. The score $\sum_i(y_i-\mu_i)\mathbf{x}_i$ has mean $\mathbf{0}$ because the **mean** is correctly specified, and variance $\sum_i\phi\mu_i\mathbf{x}_i\mathbf{x}_i'$, so its limiting average variance is $\phi\mathbf{F}$; the Lyapunov corollary applies with $\delta=2$ under the assumed fourth-moment bound. Part (iii) of that theorem gives the sandwich $\mathbf{G}^{-1}(\phi\mathbf{F})\mathbf{G}^{-1}=\phi\mathbf{F}^{-1}$.

(iii) $\hat\phi\xrightarrow{\,p\,}\phi$ by Proposition 8.17, $\sqrt n(\mathbf b-\boldsymbol{\beta}_0)$ being bounded in probability by (ii). By part (v) of the theorem, $n^{-1}\mathcal{I}(\mathbf b)\xrightarrow{\,p\,}\mathbf{F}$, so $\hat\phi\,\mathcal{I}(\mathbf b)^{-1}$ estimates $\phi\mathbf{F}^{-1}/n$ consistently. ∎

**Proposition 8.20** (Robust standard errors; FREES §12.3). If only $\operatorname{\mathsf{E}} y_i=\mu_i=E_ie^{\mathbf{x}_i'\boldsymbol{\beta}_0}$ is assumed, with $\operatorname{var} y_i=\tau_i^2$ unrestricted, $\sup_i\operatorname{\mathsf{E}}|y_i-\mu_i|^{4}<\infty$, (D1), (D2), and $n^{-1}\sum_i\tau_i^2\mathbf{x}_i\mathbf{x}_i'\to\mathbf{F}_\tau$ positive definite, then $\sqrt n(\mathbf b-\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{F}^{-1}\mathbf{F}_\tau\mathbf{F}^{-1})$, consistently estimated after division by $n$ by

$$
\widehat{\operatorname{var}}\,\mathbf b
=\Bigl(\sum_i\hat\mu_i\mathbf{x}_i\mathbf{x}_i'\Bigr)^{-1}
 \Bigl(\sum_i(y_i-\hat\mu_i)^2\mathbf{x}_i\mathbf{x}_i'\Bigr)
 \Bigl(\sum_i\hat\mu_i\mathbf{x}_i\mathbf{x}_i'\Bigr)^{-1} . \tag{7}
$$

**Proof.** The proof of Theorem 8.19(ii) runs unchanged with score variance $\sum_i\tau_i^2\mathbf{x}_i\mathbf{x}_i'$, giving the stated sandwich. For the middle matrix, $n^{-1}\sum_ie_i^2\mathbf{x}_i\mathbf{x}_i'$ has mean $n^{-1}\sum_i\tau_i^2\mathbf{x}_i\mathbf{x}_i'$ and entries of variance at most $M^4\sup_i\operatorname{\mathsf{E}} e_i^4/n$, so it converges to $\mathbf{F}_\tau$ by Chebyshev's inequality. Replacing $e_i$ by $e_i-\delta_i$ changes it by at most $M^2(2\Delta A_n+\Delta^2)$ in each entry, $A_n=n^{-1}\sum_i|e_i|$, with $\Delta\xrightarrow{\,p\,}0$ as in Proposition 8.17. As $\operatorname{\mathsf{E}} A_n\leqslant\sup_i(\operatorname{\mathsf{E}} e_i^4)^{1/4}$, Markov's inequality gives $\operatorname{\mathsf{P}}(\Delta A_n>\varepsilon)\leqslant\operatorname{\mathsf{E}} A_n/c+\operatorname{\mathsf{P}}(\Delta>\varepsilon/c)$ for every $c>0$, so the change tends to $0$ in probability. The outer matrices are $n\cdot n^{-1}\mathcal{I}(\mathbf b)$, handled by part (v) of the prerequisites' theorem **concave maximum likelihood**, as in the proof of Theorem 8.19(iii). ∎

**Remark** (FREES §12.3). FREES prints the quasi-likelihood variance as $(\hat\phi\sum_i\hat\mu_i\mathbf{x}_i\mathbf{x}_i')^{-1}$, which divides by $\hat\phi$; Theorem 8.19(iii) multiplies, $\hat\phi(\sum_i\hat\mu_i\mathbf{x}_i\mathbf{x}_i')^{-1}$. Quasi-Poisson forces the ratio of robust to model-based standard error to be $\sqrt{\hat\phi}$ for every coefficient; (7) does not, and a wide spread of those ratios is evidence against $\operatorname{var} y_i=\phi\mu_i$ itself. A likelihood ratio statistic computed from a quasi-Poisson fit is twice a difference of criterion values and is divided by $\hat\phi$ before comparison with $\chi^2$. Indeed, Step 6 of the proof of the prerequisites' theorem **concave maximum likelihood** gives $2\{\ell_n(\mathbf b)-\ell_n(\boldsymbol{\beta}_0)\}=\mathbf{Z}_n'\mathbf{F}^{-1}\mathbf{Z}_n+o_p(1)$ without using that the score variance equals $\mathbf{G}$; here $\mathbf{G}=\mathbf{F}$ and $\mathbf{Z}_n=n^{-1/2}\sum_i(y_i-\mu_i)\mathbf{x}_i\xrightarrow{\,d\,} Z\sim N(\mathbf{0},\phi\mathbf{F})$ by the proof of Theorem 8.19(ii). Fixing $q$ coordinates of $\boldsymbol{\beta}$ at their true values absorbs them into the exposures, which keeps (D1) and (D2) with $\mathbf{F}$ replaced by its leading block $\mathbf{F}_{11}$, so the restricted fit has the same expansion with $\mathbf{Z}_{n,1}$ and $\mathbf{F}_{11}$. The algebra of the proof of the prerequisites' theorem **likelihood ratio**, applied to $Z/\sqrt\phi\sim N(\mathbf{0},\mathbf{F})$, then shows that twice the difference of the two criterion values converges in law to $\phi$ times a $\chi^2_q$ variable, and Slutsky's theorem with $\hat\phi\xrightarrow{\,p\,}\phi$ (Proposition 8.17, its hypothesis supplied by Theorem 8.19(ii)) gives convergence of the ratio to $\chi^2_q$.

**Example 8.9** (Overdispersion in the two data sets; FREES §12.3, ISLR §4.6.2).

In [ ]:
sandwich_ratio <- function(m) {
  X <- model.matrix(m)[, !is.na(coef(m))]; mu <- fitted(m); e <- m$y - mu
  G <- crossprod(X, mu * X); Fm <- crossprod(X, e^2 * X)
  sqrt(diag(solve(G, Fm) %*% solve(G))) / sqrt(diag(solve(G))) }
phi_hat <- function(m) sum((m$y - fitted(m))^2 / fitted(m)) / df.residual(m)
q3 <- glm(fS, offset = LNWEIGHT, family = quasipoisson, data = S)
qB <- glm(fB, data = B, family = quasipoisson)
rbind(singapore = c(phi_hat = phi_hat(m3), sqrt_phi = sqrt(phi_hat(m3)),
                    range(sandwich_ratio(m3))),
      bikeshare = c(phi_hat(mpo), sqrt(phi_hat(mpo)), range(sandwich_ratio(mpo))))
c(coef_diff = max(abs(coef(q3) - coef(m3)), na.rm = TRUE),
  se_ratio_minus_sqrt_phi = max(abs(coef(summary(qB))[, 2] / coef(summary(mpo))[, 2] -
                                    sqrt(summary(qB)$dispersion))))

On the Singapore fit $\hat\phi=0.997$ and the robust ratios lie between $0.94$ and $1.07$: given the covariates and the exposure, nothing contradicts $\operatorname{var} y_i=\mu_i$. On `Bikeshare`, $\hat\phi=25.6$, so every Poisson standard error of Example 8.4 is too small by the factor $5.06$ under (6), and the robust ratios run from $0.16$ to $7.86$, so $\operatorname{var} y_i=\phi\mu_i$ fails as well. The quasi-Poisson coefficients equal the Poisson ones and its standard errors are exactly $\sqrt{\hat\phi}$ times as large, Theorem 8.19(i) and (iii).

### 4.2 The negative binomial

**Definition 8.21** (Negative binomial; FREES (12.10), Table 13.8). For $r>0$ and $p\in(0,1)$, $y\sim\mathrm{NB}(r,p)$ if

$$
\operatorname{\mathsf{P}}(y=j)=\frac{\Gamma(j+r)}{j!\,\Gamma(r)}\,p^r(1-p)^j , \qquad j=0,1,2,\ldots , \tag{8}
$$

which is $\binom{j+r-1}{r-1}p^r(1-p)^j$ for integer $r$. With $\mu=r(1-p)/p$ and $\sigma=1/r$ the same law is written $\mathrm{NB}(\mu,\sigma)$; then $p=1/(1+\sigma\mu)$.

**Theorem 8.22** (Poisson–gamma mixture; FREES §12.3). Let $\lambda$ have the gamma density with shape $r$ and rate $r/\mu$, so that $\operatorname{\mathsf{E}}\lambda=\mu$ and $\operatorname{var}\lambda=\mu^2/r$, and let $y$ given $\lambda$ be $\mathrm{Pois}(\lambda)$. Then $y\sim\mathrm{NB}(r,p)$ with $p=r/(r+\mu)$, and

$$
\operatorname{\mathsf{E}} y=\mu=\frac{r(1-p)}{p} , \qquad \operatorname{var} y=\mu+\frac{\mu^2}{r}=\frac{r(1-p)}{p^2} . \tag{9}
$$

In particular (8) sums to $1$ for every $r>0$, $p\in(0,1)$.

**Proof.** Write $\rho=r/\mu$. By Fubini–Tonelli,

$$
\operatorname{\mathsf{P}}(y=j)=\int_0^\infty\frac{\lambda^je^{-\lambda}}{j!}\,
\frac{\rho^r}{\Gamma(r)}\lambda^{r-1}e^{-\rho\lambda}\,\mathrm{d}\lambda
=\frac{\rho^r}{j!\,\Gamma(r)}\int_0^\infty\lambda^{j+r-1}e^{-(1+\rho)\lambda}\,\mathrm{d}\lambda ,
$$

and the substitution $u=(1+\rho)\lambda$ turns the integral into $\Gamma(j+r)/(1+\rho)^{j+r}$. Since $\rho/(1+\rho)=r/(r+\mu)=p$ and $1/(1+\rho)=1-p$, this is (8). Every $p\in(0,1)$ arises from $\mu=r(1-p)/p$, so (8) is a mass function. The tower property and the conditional variance decomposition of the prerequisites give $\operatorname{\mathsf{E}} y=\operatorname{\mathsf{E}}\lambda=\mu$ and $\operatorname{var} y=\operatorname{\mathsf{E}}\lambda+\operatorname{var}\lambda=\mu+\mu^2/r$; substituting $\mu=r(1-p)/p$ gives the $(r,p)$ forms. ∎

**Corollary 8.23** (The Poisson is a limit). For fixed $\mu$ and $j$, $\operatorname{\mathsf{P}}(y=j)\to\mu^je^{-\mu}/j!$ as $r\to\infty$, that is, as $\sigma\to0$ with $p=r/(r+\mu)\to1$; and $\operatorname{var} y\downarrow\mu$.

**Proof.** The variance is immediate from (9). Write (8) with $p=r/(r+\mu)$ as

$$
\frac{\Gamma(j+r)}{\Gamma(r)r^j}\cdot\frac{\mu^j}{j!}\cdot\Bigl(1+\frac\mu r\Bigr)^{-r}
\Bigl(1+\frac\mu r\Bigr)^{-j} .
$$

The first factor is $\prod_{l=0}^{j-1}(r+l)/r\to1$, using $\Gamma(x+1)=x\Gamma(x)$; the third tends to $e^{-\mu}$ and the fourth to $1$. ∎

**Remark** (FREES §12.3). FREES's limit “$p\to1$ and $r\to0$ such that $rp\to\lambda$” is degenerate: $rp\leqslant r\to0$ forces $\lambda=0$, and with $r\to0$ at fixed mean the variance $\mu+\mu^2/r$ diverges. The Poisson is the limit $r\to\infty$ of Corollary 8.23, on the boundary of the parameter space $\sigma\in(0,\infty)$ rather than inside it; the prerequisites' theorem **likelihood ratio** assumes an interior null value, so it does not license a $\chi^2_1$ test of Poisson against negative binomial, as FREES notes.

**Definition 8.24** (Negative binomial regression; FREES §12.3). The $y_i$ are independent $\mathrm{NB}(\mu_i,\sigma)$ with $\ln\mu_i=\ln E_i+\mathbf{x}_i'\boldsymbol{\beta}$ and one $\sigma>0$; by (9), $\operatorname{var} y_i=\mu_i+\sigma\mu_i^2$.

**Proposition 8.25** (Concavity in $\boldsymbol{\beta}$ for fixed $\sigma$). For fixed $\sigma$ the negative binomial log-likelihood is concave in $\boldsymbol{\beta}$, strictly when $\operatorname{rank}\mathbf{X}=k+1$, with score $\sum_i(y_i-\mu_i)\mathbf{x}_i/(1+\sigma\mu_i)$.

**Proof.** With $r=1/\sigma$, the terms of $\ln\operatorname{\mathsf{P}}(y_i=j)$ involving $\mu_i$ are $y_i\ln\mu_i-(y_i+r)\ln(r+\mu_i)$. As a function of $\eta=\ln\mu_i$, with $\mathrm{d}\mu_i/\mathrm{d}\eta=\mu_i$, the first derivative is $y_i-(y_i+r)\mu_i/(r+\mu_i)=r(y_i-\mu_i)/(r+\mu_i)=(y_i-\mu_i)/(1+\sigma\mu_i)$ and the second is $-(y_i+r)r\mu_i/(r+\mu_i)^2<0$ (the prerequisites' lemma **convexity from the derivative**). Composing with the linear map $\boldsymbol{\beta}\mapsto\ln E_i+\mathbf{x}_i'\boldsymbol{\beta}$ gives the score and preserves concavity, and the Hessian $-\sum_i\{(y_i+r)r\mu_i/(r+\mu_i)^2\}\mathbf{x}_i\mathbf{x}_i'$ is negative definite when the $\mathbf{x}_i$ span $\mathbb{R}^{k+1}$. ∎

**Example 8.10** (Singapore, negative binomial; FREES §12.3, Table 12.7). FREES's script fits the Table 12.6 model by `MASS::glm.nb`, whose `theta` is $r$:

In [ ]:
nb3 <- glm.nb(update(fS, . ~ . + offset(LNWEIGHT)), data = S)
print(c(r = nb3$theta, se_r = nb3$SE.theta, logLik = as.numeric(logLik(nb3)),
        logLik_frees_convention = as.numeric(logLik(nb3)) + lfact,
        AIC_poisson = AIC(m3), AIC_negbin = AIC(nb3)), digits = 7)
Pnb <- function(mu, r) sapply(0:4, function(j) dnbinom(j, size = r, mu = mu))
rbind(negbin = cells(Pnb(fitted(nb3), nb3$theta)),
      frees_size_2.33 = cells(Pnb(fitted(nb3), 2.33)))

$\hat r=2.33$ and, in FREES's convention without $\sum\ln y_i!$, the log-likelihood $-1774.494$ of FREES §12.3 are reproduced. FREES's negative binomial column of Table 12.7 ($6996.04$, $453.40$, $31.09$, $2.28$, $0.18$; statistic $1.79$) is not: the model of its script gives $6997.00$, $451.65$, $31.71$, $2.42$, $0.20$ and statistic $1.69$, and the script's fixed `size = 2.33` changes the second decimal only. With $\hat\mu_i$ near $0.07$, $\sigma\hat\mu_i^2$ is small beside $\hat\mu_i$, yet the extra variance moves fitted mass from $1$ to $0$, $2$ and $3$, which is where the Poisson column missed.

## 5 Other Count Models

> **Reading.** FREES §12.4. SRM learning outcomes 2(a), 2(e).

### 5.1 Zero-inflated and hurdle models

**Definition 8.26** (Zero-inflated and hurdle models; FREES (12.11)–(12.12)). Let $g_i$ be a count mass function with mean $\mu_i$ and $\pi_i\in(0,1)$.

(i) The **zero-inflated** model mixes a point mass at $0$ with $g_i$:

$$
\operatorname{\mathsf{P}}(y_i=j)=\begin{cases}\pi_i+(1-\pi_i)g_i(0), & j=0,\\
                               (1-\pi_i)g_i(j), & j\geqslant1.\end{cases} \tag{10}
$$

(ii) The **hurdle** model fixes the zero probability and rescales the rest, for $g_i(0)<1$:

$$
\operatorname{\mathsf{P}}(y_i=j)=\begin{cases}\pi_i, & j=0,\\ k_i\,g_i(j), & j\geqslant1,\end{cases}
      \qquad k_i=\frac{1-\pi_i}{1-g_i(0)} . \tag{11}
$$

In both, $\pi_i$ is usually a logistic regression and $\mu_i$ a log-linear one.

**Theorem 8.27** (Moments; FREES §12.4.1–12.4.2). Let $g_i$ be the $\mathrm{Pois}(\mu_i)$ mass function.

(i) Under (10), $\operatorname{\mathsf{E}} y_i=(1-\pi_i)\mu_i$ and $\operatorname{var} y_i=(1-\pi_i)\mu_i+\pi_i(1-\pi_i)\mu_i^2$, which exceeds the mean.

(ii) Under (11), $\operatorname{\mathsf{E}} y_i=k_i\mu_i$ and $\operatorname{var} y_i=k_i\mu_i+k_i(1-k_i)\mu_i^2$, which exceeds the mean if $k_i<1$ and falls below it if $k_i>1$.

**Proof.** (i) Let $D$ be a Bernoulli$(\pi_i)$ indicator independent of $N\sim\mathrm{Pois}(\mu_i)$; then $(1-D)N$ has law (10). As $(1-D)^2=1-D$, $\operatorname{\mathsf{E}} y_i=(1-\pi_i)\mu_i$ and $\operatorname{\mathsf{E}} y_i^2=(1-\pi_i)(\mu_i+\mu_i^2)$, so $\operatorname{var} y_i=(1-\pi_i)\mu_i+(1-\pi_i)\mu_i^2-(1-\pi_i)^2\mu_i^2$, and the last two terms combine to $\pi_i(1-\pi_i)\mu_i^2>0$.

(ii) The $j=0$ terms contribute nothing to $\operatorname{\mathsf{E}} y_i=k_i\sum_{j\geqslant1}jg_i(j) =k_i\mu_i$ or to $\operatorname{\mathsf{E}} y_i^2=k_i(\mu_i+\mu_i^2)$; subtract the squared mean. ∎

**Remark** (FREES §12.4.1). FREES prints the zero-inflated Poisson variance as $\pi_i\mu_i+\pi_i\mu_i^2(1-\pi_i)$; the first term is $(1-\pi_i)\mu_i$, as in Theorem 8.27(i). FREES's conclusion, that the variance exceeds the mean, holds for the corrected formula.

**Theorem 8.28** (The hurdle likelihood separates; FREES §12.4.2). Let $\pi_i=\pi(\mathbf{x}_i;\boldsymbol{\gamma})$ and $g_i=g(\cdot;\mathbf{x}_i,\boldsymbol{\beta})$ with $\boldsymbol{\gamma}$ and $\boldsymbol{\beta}$ variation-independent. The log-likelihood of (11) is

$$
L(\boldsymbol{\gamma},\boldsymbol{\beta})=
\underbrace{\sum_i\bigl[\mathbb{1}\{y_i=0\}\ln\pi_i+\mathbb{1}\{y_i>0\}\ln(1-\pi_i)\bigr]}_{L_1(\boldsymbol{\gamma})}
+\underbrace{\sum_{i:y_i>0}\ln\frac{g_i(y_i)}{1-g_i(0)}}_{L_2(\boldsymbol{\beta})} ,
$$

so it is maximised by maximising the binary log-likelihood $L_1$ and the zero-truncated count log-likelihood $L_2$ separately.

**Proof.** An observation with $y_i=0$ contributes $\ln\pi_i$, and one with $y_i>0$ contributes $\ln k_i+\ln g_i(y_i)=\ln(1-\pi_i)+\ln\{g_i(y_i)/(1-g_i(0))\}$. No parameter appears in both sums, so $\sup L=\sup L_1+\sup L_2$, attained at any pair of separate maximisers. ∎

**Corollary 8.29** (An intercept-only hurdle reproduces the zeros). If $\pi_i=e^{\gamma_0}/(1+e^{\gamma_0})$ for all $i$, with $0<\#\{y_i=0\}<n$, then $\hat\pi=n^{-1}\#\{i:y_i=0\}$ and the fitted number of zeros equals the observed number.

**Proof.** $L_1$ is $N_0\ln\pi+(n-N_0)\ln(1-\pi)$ with $N_0=\#\{y_i=0\}$, composed with the strictly increasing $\gamma_0\mapsto\pi$; its derivative in $\pi$, $N_0/\pi-(n-N_0)/(1-\pi)$, is strictly decreasing and vanishes at $\pi=N_0/n$, the maximiser, and the fitted number of zeros is $n\hat\pi=N_0$. ∎

### 5.2 Heterogeneity and latent class models

**Proposition 8.30** (Heterogeneity; FREES §12.4.3). Let $y_i$ given $\alpha_i$ be $\mathrm{Pois}(e^{\alpha_i}\mu_i)$, where $\operatorname{\mathsf{E}} e^{\alpha_i}=1$ and $\operatorname{var} e^{\alpha_i}=\tau^2<\infty$. Then $\operatorname{\mathsf{E}} y_i=\mu_i$ and $\operatorname{var} y_i=\mu_i+\tau^2\mu_i^2$. If $e^{\alpha_i}$ is gamma with shape $r$ and rate $r$, $y_i\sim\mathrm{NB}(\mu_i,1/r)$.

**Proof.** By the tower property and the conditional variance decomposition, $\operatorname{\mathsf{E}} y_i=\mu_i\operatorname{\mathsf{E}} e^{\alpha_i}=\mu_i$ and $\operatorname{var} y_i=\operatorname{\mathsf{E}}(e^{\alpha_i}\mu_i)+\operatorname{var}(e^{\alpha_i}\mu_i)=\mu_i+\tau^2\mu_i^2$. In the gamma case $\lambda=e^{\alpha_i}\mu_i$ is gamma with shape $r$ and rate $r/\mu_i$, and Theorem 8.22 applies. ∎

**Proposition 8.31** (Latent classes; FREES (12.13)). Let $\operatorname{\mathsf{P}}(y=j)=\sum_{c=1}^m\pi_c\operatorname{\mathsf{P}}_c(y=j)$ with weights $\pi_c>0$ summing to $1$ and component means $\mu_c$ and variances $v_c$. Then $\operatorname{var} y=\sum_c\pi_cv_c+\sum_c\pi_c(\mu_c-\bar\mu)^2$ with $\bar\mu=\sum_c\pi_c\mu_c$. If every component has $v_c\geqslant\mu_c$, as the Poisson and the negative binomial do, then $\operatorname{var} y\geqslant\operatorname{\mathsf{E}} y$, strictly unless every $v_c=\mu_c$ and all $\mu_c$ are equal.

**Proof.** Let $C$ be the class, with $\operatorname{\mathsf{P}}(C=c)=\pi_c$. The conditional variance decomposition gives $\operatorname{var} y=\operatorname{\mathsf{E}} v_C+\operatorname{var}\mu_C$, which is the displayed sum, and $\operatorname{\mathsf{E}} y=\bar\mu$. Then $\operatorname{var} y-\operatorname{\mathsf{E}} y=\sum_c\pi_c(v_c-\mu_c)+\sum_c\pi_c(\mu_c-\bar\mu)^2$, a sum of non-negative terms. ∎

**Remark** (FREES §12.4.4). FREES says a latent class model “readily accommodates under- and overdispersion”. With Poisson or negative binomial classes, as in its examples, it cannot produce underdispersion (Proposition 8.31); among the models of this section only the hurdle can (Theorem 8.27(ii)).

**Example 8.11** (Excess zeros in the Singapore data; FREES §12.4). The Table 12.6 count part with an intercept-only zero part, fitted by `pscl`:

In [ ]:
suppressMessages(library(pscl))
S$X3 <- model.matrix(m3)[, !is.na(coef(m3))][, -1]
zi <- zeroinfl(Clm_Count ~ X3 + offset(LNWEIGHT) | 1, data = S, dist = "poisson")
hu <- hurdle(Clm_Count ~ X3 + offset(LNWEIGHT) | 1, data = S, dist = "poisson")
round(c(AIC_poisson = AIC(m3), AIC_negbin = AIC(nb3), AIC_zip = AIC(zi),
        AIC_hurdle = AIC(hu), pi_zip = plogis(coef(zi)[["zero_(Intercept)"]])), 3)
round(c(observed = sum(y == 0), poisson = sum(dpois(0, fitted(m3))),
        negbin = sum(dnbinom(0, size = nb3$theta, mu = fitted(nb3))),
        zip = sum(predict(zi, type = "prob")[, 1]),
        hurdle = sum(predict(hu, type = "prob")[, 1])), 2)

The hurdle's fitted zeros equal the observed $6996$ exactly, Corollary 8.29; its $\mathrm{AIC}$ is the worst, $232$ above the zero-inflated model's, because a zero probability without covariates or exposure ignores that a policy in force for a month has far fewer claims than one in force for a year. The zero-inflated Poisson places $\hat\pi=0.279$ of the policies in the point mass and has $\mathrm{AIC}$ within $0.7$ of the negative binomial.

## 6 Generalised Linear Models

> **Reading.** FREES §13.1–13.3 (and §13.9, beyond the syllabus); ISLR §4.6.3 (PA reading list, not SRM). SRM learning outcomes 2(a), 2(b), 2(e).

### 6.1 The linear exponential family

**Definition 8.32** (Linear exponential family; FREES (13.2)). Fix $\phi>0$, a $\sigma$-finite measure $\lambda$ on $\mathbb{R}$ and a function $S(\cdot,\phi)$ such that $\nu(\mathrm{d} y)=e^{S(y,\phi)}\lambda(\mathrm{d} y)$ is not concentrated at one point. Let $\Theta$ be the set of $\theta$ with $A(\theta)=\int e^{y\theta/\phi}\,\nu(\mathrm{d} y)<\infty$, assume its interior $\Theta^\circ$ is non-empty, and put $b(\theta)=\phi\ln A(\theta)$. The **linear exponential family** is the set of densities with respect to $\lambda$

$$
f(y;\theta,\phi)=\exp\Bigl\{\frac{y\theta-b(\theta)}{\phi}+S(y,\phi)\Bigr\} ,
\qquad \theta\in\Theta^\circ , \tag{12}
$$

with **natural parameter** $\theta$ and **scale parameter** $\phi$. Defining $b$ from $A$ makes $\int f\,\mathrm{d}\lambda=1$ hold by construction.

**Theorem 8.33** (Moments; FREES §13.2.1 and, beyond the syllabus, §13.9.2). $b$ has derivatives of every order on $\Theta^\circ$, and for $y$ with density (12)

$$
\operatorname{\mathsf{E}} y=b'(\theta) , \qquad \operatorname{var} y=\phi\,b''(\theta)>0 . \tag{13}
$$

So $b$ is strictly convex (the prerequisites' lemma **convexity from the derivative**) and $b'$ strictly increasing on $\Theta^\circ$.

**Proof.** Fix $\theta_0\in\Theta^\circ$ and $\delta>0$ with $[\theta_0-2\delta,\theta_0+2\delta]\subset\Theta$.

**Domination.** For $k\geqslant0$ and $u\geqslant0$, $e^u\geqslant u^k/k!$, so $|y|^k\leqslant k!(\phi/\delta)^ke^{|y|\delta/\phi}$. For $|\theta-\theta_0|<\delta$ the exponent of $e^{|y|\delta/\phi}e^{y\theta/\phi}$ is at most $y(\theta_0+2\delta)/\phi$ when $y\geqslant0$ and at most $y(\theta_0-2\delta)/\phi$ when $y<0$. Hence

$$
|y|^ke^{y\theta/\phi}\leqslant k!(\phi/\delta)^k\bigl\{e^{y(\theta_0+2\delta)/\phi}
+e^{y(\theta_0-2\delta)/\phi}\bigr\}=:g_k(y) ,
$$

$\nu$-integrable because $\theta_0\pm2\delta\in\Theta$.

**Differentiation.** The $k$th $\theta$-derivative of $e^{y\theta/\phi}$ is $(y/\phi)^ke^{y\theta/\phi}$, dominated by $g_k/\phi^k$ on $(\theta_0-\delta,\theta_0+\delta)$. By induction on $k$ with the prerequisites' theorem **differentiation under the integral sign**, $A$ has derivatives of every order there, $A^{(k)}(\theta)=\phi^{-k}\int y^ke^{y\theta/\phi}\,\nu(\mathrm{d} y)$, each continuous because differentiable. As $A>0$, $b=\phi\ln A$ has derivatives of every order. Since $f=e^{y\theta/\phi}e^{S}/A$, $A'=A\operatorname{\mathsf{E}} y/\phi$ and $A''=A\operatorname{\mathsf{E}} y^2/\phi^2$, so

$$
b'=\phi\frac{A'}{A}=\operatorname{\mathsf{E}} y , \qquad
b''=\phi\,\frac{A''A-(A')^2}{A^2}=\frac{\operatorname{\mathsf{E}} y^2-(\operatorname{\mathsf{E}} y)^2}{\phi} .
$$

The law of $y$ is not concentrated at a point, so $\operatorname{var} y>0$. ∎

**Definition 8.34** (Variance function, weights; FREES §13.1–13.2). By Theorem 8.33, $\theta\mapsto\mu=b'(\theta)$ is a strictly increasing bijection of $\Theta^\circ$ onto its range, and

$$
\operatorname{var} y=\phi\,v(\mu) , \qquad v(\mu)=b''\bigl((b')^{-1}(\mu)\bigr) , \tag{14}
$$

the **variance function**. Observation $i$ with a known **weight** $w_i>0$ has scale $\phi_i=\phi/w_i$, so $\operatorname{var} y_i=\phi v(\mu_i)/w_i$.

**Proposition 8.35** (Members; FREES Tables 13.1, 13.2, 13.8). With $\lambda$ Lebesgue measure on $\mathbb{R}$ or $(0,\infty)$, or counting measure on the support, each law below is (12) with the components shown.

| Law | $\theta$ | $\phi$ | $b(\theta)$ | $S(y,\phi)$ | $v(\mu)$ |
|:--|:-:|:-:|:-:|:-:|:-:|
| $N(\mu,\sigma^2)$ | $\mu$ | $\sigma^2$ | $\theta^2/2$ | $-\frac{y^2}{2\phi}-\frac12\ln(2\pi\phi)$ | $1$ |
| Binomial$(m,\pi)$ | $\operatorname{logit}\pi$ | $1$ | $m\ln(1+e^\theta)$ | $\ln\binom my$ | $\mu(1-\mu/m)$ |
| $\mathrm{Pois}(\mu)$ | $\ln\mu$ | $1$ | $e^\theta$ | $-\ln y!$ | $\mu$ |
| NB$(r,p)$, $r$ known | $\ln(1-p)$ | $1$ | $-r\ln(1-e^\theta)$ | $\ln\frac{\Gamma(y+r)}{y!\Gamma(r)}$ | $\mu+\mu^2/r$ |
| Gamma$(\alpha,\gamma)$ | $-\gamma/\alpha$ | $1/\alpha$ | $-\ln(-\theta)$ | $-\frac{\ln\phi}{\phi}-\ln\Gamma(\frac1\phi)+(\frac1\phi-1)\ln y$ | $\mu^2$ |
| IG$(\mu,\lambda_0)$ | $-\frac1{2\mu^2}$ | $1/\lambda_0$ | $-\sqrt{-2\theta}$ | $-\frac1{2\phi y}-\frac12\ln(2\pi\phi y^3)$ | $\mu^3$ |

Here the gamma density is $\gamma^\alpha y^{\alpha-1}e^{-\gamma y}/\Gamma(\alpha)$ and the inverse Gaussian density is $\sqrt{\lambda_0/(2\pi y^3)}\exp\{-\lambda_0(y-\mu)^2/(2\mu^2y)\}$.

**Proof.** In each case expand $\ln f$ and match. Normal: $-(y-\mu)^2/(2\sigma^2)=(y\mu-\mu^2/2)/\sigma^2 -y^2/(2\sigma^2)$. Binomial: $y\ln\pi+(m-y)\ln(1-\pi)=y\operatorname{logit}\pi+m\ln(1-\pi)$ and $\ln(1-\pi)=-\ln(1+e^\theta)$. Poisson: $y\ln\mu-\mu$. Negative binomial: $y\ln(1-p)+r\ln p$ with $p=1-e^\theta$. Gamma: with $\theta=-\gamma/\alpha$ and $\phi=1/\alpha$, $(y\theta-b)/\phi=-\gamma y+\alpha\ln\gamma-\alpha\ln\alpha$, and $\ln f$ minus this is $(\alpha-1)\ln y-\ln\Gamma(\alpha)+\alpha\ln\alpha$, which is the stated $S$. Inverse Gaussian: $-\lambda_0(y-\mu)^2/(2\mu^2y)=\lambda_0\{-y/(2\mu^2)+1/\mu\}-\lambda_0/(2y)$, and $(y\theta-b)/\phi$ is the braced part times $\lambda_0$. In each row $v(\mu)$ is $b''$ at $\theta=(b')^{-1}(\mu)$ by Theorem 8.33: for the gamma $b'=-1/\theta=\mu$ and $b''=1/\theta^2=\mu^2$; for the inverse Gaussian $b'=(-2\theta)^{-1/2}=\mu$ and $b''=(-2\theta)^{-3/2}=\mu^3$; for the negative binomial $b'=re^\theta/(1-e^\theta)=\mu$ and $b''=b'/(1-e^\theta)=\mu(1+\mu/r)$. ∎

**Remark** (FREES Table 13.8, §13.1). FREES's gamma entry has $S(y,\phi)=+\phi^{-1}\ln\phi-\cdots$; the sign is negative, and FREES's version multiplies every density by $e^{2\phi^{-1}\ln\phi}=\phi^{2/\phi}$, which is $0.4^5=0.010$ at $\alpha=2.5$. Its inverse Gaussian entry has $\theta/(\phi y)$ in place of $-1/(2\phi y)$, a term that depends on $\mu$. In §13.1, FREES writes $\mathbf{x}_i'\boldsymbol{\beta}=\exp(\mu_i)/(1+\exp(\mu_i))$ for logistic regression; the systematic component is $\mathbf{x}_i'\boldsymbol{\beta}=\operatorname{logit}\mu_i$. ISLR §4.6.3 lists the negative binomial among exponential families, which holds with $r$ known, as in the table.

### 6.2 Link functions

**Definition 8.36** (Generalised linear model; FREES (13.1), ISLR (4.42)). A **generalised linear model** has independent $y_1,\ldots,y_n$ from one linear exponential family with weights $w_i$ and natural parameters $\theta_i$, a strictly monotone differentiable **link** $g$ with $g'\neq0$, and the **systematic component**

$$
\eta_i=\mathbf{x}_i'\boldsymbol{\beta}=g(\mu_i) , \qquad \mu_i=b'(\theta_i) . \tag{15}
$$

The link is **canonical** when $g=(b')^{-1}$, that is when $\theta_i=\eta_i$. ISLR's linear, logistic and Poisson regressions are the normal, Bernoulli and Poisson members with their canonical links $\mu$, $\operatorname{logit}\mu$ and $\ln\mu$.

**Remark** (Links for the Bernoulli; FREES §13.2.2). A Bernoulli mean lies in $(0,1)$, and the logit $\ln\{\mu/(1-\mu)\}$, the probit $\Phi^{-1}(\mu)$ and the complementary log-log $\ln\{-\ln(1-\mu)\}$ map $(0,1)$ onto $\mathbb{R}$, so every $\mathbf{x}_i'\boldsymbol{\beta}$ gives a mean in $(0,1)$; the identity and the logarithm do not. Under the logarithmic link, Proposition 8.10(iv) gives a multiplicative rating plan $\mu=B\cdot A_i\cdot T_j$; under the identity link the plan is additive. A link is often written up to a constant factor ($1/\mu^2$ for the inverse Gaussian), which rescales $\boldsymbol{\beta}$.

### 6.3 Estimation

**Proposition 8.37** (Canonical links; FREES §13.3.1). Under the canonical link,

$$
L(\boldsymbol{\beta},\phi)=\sum_i\Bigl\{\frac{w_i\{y_i\mathbf{x}_i'\boldsymbol{\beta}-b(\mathbf{x}_i'\boldsymbol{\beta})\}}{\phi}
+S(y_i,\phi/w_i)\Bigr\} ,
$$

the score is $\phi^{-1}\sum_iw_i(y_i-\mu_i)\mathbf{x}_i$, and $-\nabla^2L=\phi^{-1}\sum_iw_i b''(\mathbf{x}_i'\boldsymbol{\beta})\mathbf{x}_i\mathbf{x}_i'=\mathcal{I}(\boldsymbol{\beta})$ whatever the responses. $L$ is concave in $\boldsymbol{\beta}$, strictly when $\operatorname{rank}\mathbf{X}=k+1$, and the maximiser solves the normal equations $\sum_iw_i(y_i-\mu_i)\mathbf{x}_i=\mathbf{0}$, which do not involve $\phi$.

**Proof.** Substitute $\theta_i=\mathbf{x}_i'\boldsymbol{\beta}$ and $\phi_i=\phi/w_i$ into (12). Differentiating, $\nabla L=\phi^{-1}\sum_iw_i\{y_i-b'(\mathbf{x}_i'\boldsymbol{\beta})\}\mathbf{x}_i$ and $\nabla^2L=-\phi^{-1}\sum_iw_ib''(\mathbf{x}_i'\boldsymbol{\beta})\mathbf{x}_i\mathbf{x}_i'$, free of the $y_i$, hence equal to its expectation, which is $-\operatorname{var}\nabla L$ by Definition 8.34. $b''>0$ (Theorem 8.33) makes $-\nabla^2L$ positive semi-definite, and positive definite when the $\mathbf{x}_i$ span, which gives the concavity claims by the prerequisites' lemma **convexity from the derivative**. ∎

**Theorem 8.38** (Score and information, general link; FREES (13.13)–(13.14)). For any link,

$$
\nabla L(\boldsymbol{\beta})=\sum_{i=1}^n\frac{\partial\mu_i}{\partial\boldsymbol{\beta}}\,\frac{y_i-\mu_i}{\operatorname{var} y_i}
=\sum_{i=1}^n\frac{w_i}{\phi}\,\frac{y_i-\mu_i}{v(\mu_i)}\,\frac{\mathrm{d}\mu_i}{\mathrm{d}\eta_i}\,\mathbf{x}_i ,
\qquad
\mathcal{I}(\boldsymbol{\beta})=\sum_{i=1}^n\frac{w_i}{\phi\,v(\mu_i)}\Bigl(\frac{\mathrm{d}\mu_i}{\mathrm{d}\eta_i}\Bigr)^{2}
\mathbf{x}_i\mathbf{x}_i' .
$$

**Proof.** $\partial L/\partial\theta_i=w_i(y_i-\mu_i)/\phi$, since $b'(\theta_i)=\mu_i$. Differentiating $\mu_i=b'(\theta_i)$ gives $\mathrm{d}\mu_i/\mathrm{d}\theta_i=b''(\theta_i)=v(\mu_i)$, so by the chain rule through $\boldsymbol{\beta}\mapsto\eta_i\mapsto\mu_i\mapsto\theta_i$,

$$
\frac{\partial L}{\partial\boldsymbol{\beta}}=\sum_i\frac{w_i(y_i-\mu_i)}{\phi}\,\frac{1}{v(\mu_i)}\,
\frac{\mathrm{d}\mu_i}{\mathrm{d}\eta_i}\,\mathbf{x}_i ,
$$

which is both displayed forms, as $\operatorname{var} y_i=\phi v(\mu_i)/w_i$ and $\partial\mu_i/\partial\boldsymbol{\beta}=(\mathrm{d}\mu_i/\mathrm{d}\eta_i)\mathbf{x}_i$. The summands are independent with mean zero, so $\mathcal{I}=\operatorname{var}\nabla L$ is the sum of $(w_i/\phi)^2v(\mu_i)^{-2}(\mathrm{d}\mu_i/\mathrm{d}\eta_i)^2\operatorname{var} y_i\,\mathbf{x}_i\mathbf{x}_i'$; substitute $\operatorname{var} y_i$. ∎

**Corollary 8.39** (No covariates; FREES §13.9.3 (beyond the syllabus)). If $\mathbf{x}_i=1$ for all $i$, the maximum likelihood estimate of $\mu$ is the weighted mean $\bar y_w=\sum_iw_iy_i/\sum_iw_i$, whatever the family and the link, provided $\bar y_w$ lies in the range of $b'$.

**Proof.** The score is $(\mathrm{d}\mu/\mathrm{d}\eta)\{\phi v(\mu)\}^{-1}\sum_iw_i(y_i-\mu)$, with a nonzero factor in front, and $\sum_iw_i(y_i-\mu)$ is strictly decreasing in $\mu$ and zero at $\bar y_w$. Since $\mu$ is a strictly monotone function of $\beta$, $L$ increases in $\mu$ below $\bar y_w$ and decreases above it. ∎

**Theorem 8.40** (Iteratively reweighted least squares; FREES §13.9.4 (beyond the syllabus)). At the current $\boldsymbol{\beta}^{(m)}$ let

$$
W_i=\frac{w_i}{\phi\,v(\mu_i)}\Bigl(\frac{\mathrm{d}\mu_i}{\mathrm{d}\eta_i}\Bigr)^{2} ,
\qquad
z_i=\eta_i+\frac{y_i-\mu_i}{\mathrm{d}\mu_i/\mathrm{d}\eta_i} .
$$

The Fisher scoring step $\boldsymbol{\beta}^{(m+1)}=\boldsymbol{\beta}^{(m)}+\mathcal{I}(\boldsymbol{\beta}^{(m)})^{-1}\nabla L(\boldsymbol{\beta}^{(m)})$ equals the weighted least squares fit $(\mathbf{X}'\mathbf{W}\mathbf{X})^{-1}\mathbf{X}'\mathbf{W}\mathbf{z}$ with $\mathbf{W}=\operatorname{diag}(W_i)$, and does not involve $\phi$. Under the canonical link it is the Newton step.

**Proof.** By Theorem 8.38, $\mathcal{I}=\mathbf{X}'\mathbf{W}\mathbf{X}$ and $\nabla L=\sum_iW_i(z_i-\eta_i)\mathbf{x}_i=\mathbf{X}'\mathbf{W}(\mathbf{z}-\mathbf{X}\boldsymbol{\beta}^{(m)})$, because $W_i(z_i-\eta_i)=W_i(y_i-\mu_i)/(\mathrm{d}\mu_i/\mathrm{d}\eta_i)$ is the $i$th score coefficient. So $\boldsymbol{\beta}^{(m)}+(\mathbf{X}'\mathbf{W}\mathbf{X})^{-1}\mathbf{X}'\mathbf{W}(\mathbf{z}-\mathbf{X}\boldsymbol{\beta}^{(m)})=(\mathbf{X}'\mathbf{W}\mathbf{X})^{-1}\mathbf{X}'\mathbf{W}\mathbf{z}$, and the factor $1/\phi$ common to $\mathbf{W}$ cancels. Under the canonical link $\mathcal{I}=-\nabla^2L$ (Proposition 8.37). ∎

**Remark** (FREES (13.15) (beyond the syllabus)). FREES's Newton–Raphson step (13.15) subtracts $(\sum_iw_ib''\mathbf{x}_i\mathbf{x}_i')^{-1}\sum_iw_i(y_i-b')\mathbf{x}_i$ from $\boldsymbol{\beta}_{\mathrm{OLD}}$; the step adds it, as the Hessian is $-\phi^{-1}\sum_iw_ib''\mathbf{x}_i\mathbf{x}_i'$. Its final weighted least squares formula is right.

**Theorem 8.41** (Large-sample inference, canonical link; FREES §13.3.1). Consider a family with $\Theta^\circ=\mathbb{R}$ (normal, binomial, Poisson) under its canonical link, and let $\mathbf b$ maximise $\ell_n(\boldsymbol{\beta})=\sum_iw_i\{y_i\mathbf{x}_i'\boldsymbol{\beta}-b(\mathbf{x}_i'\boldsymbol{\beta})\}$. Suppose the $y_i$ are independent with $\operatorname{\mathsf{E}} y_i=b'(\mathbf{x}_i'\boldsymbol{\beta}_0)$ and $\operatorname{var} y_i=\tau_i^2$, that $|\mathbf{x}_i|\leqslant M$, $c_w\leqslant w_i\leqslant C_w$ and $\sup_i\operatorname{\mathsf{E}}(y_i-\operatorname{\mathsf{E}} y_i)^4<\infty$, and that

$$
\mathbf{G}_n=\frac1n\sum_iw_ib''(\mathbf{x}_i'\boldsymbol{\beta}_0)\mathbf{x}_i\mathbf{x}_i'\to\mathbf{G} , \qquad
\mathbf{F}_n=\frac1n\sum_iw_i^2\tau_i^2\mathbf{x}_i\mathbf{x}_i'\to\mathbf{F} ,
$$

both positive definite. Then with probability tending to one $\mathbf b$ exists and is unique, $\mathbf b\xrightarrow{\,p\,}\boldsymbol{\beta}_0$, and $\sqrt n(\mathbf b-\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\mathbf{G}^{-1}\mathbf{F}\mathbf{G}^{-1})$. If $\operatorname{var} y_i=\phi\,v(\mu_i)/w_i$, then $\mathbf{F}=\phi\mathbf{G}$ and the limit is $N(\mathbf{0},\phi\mathbf{G}^{-1})$.

**Proof.** Apply the prerequisites' theorem **concave maximum likelihood** with $\ell_i=w_i\{y_i\mathbf{x}_i'\boldsymbol{\beta}-b(\mathbf{x}_i'\boldsymbol{\beta})\}$, concave and smooth by Theorem 8.33. Its score $w_i(y_i-\mu_i)\mathbf{x}_i$ has mean $\mathbf{0}$, average variance $\mathbf{F}_n\to\mathbf{F}$, and fourth moments at most $C_w^4M^4\sup_i\operatorname{\mathsf{E}}(y_i-\mu_i)^4$, so the Lyapunov corollary with $\delta=2$ gives the normal limit of the score. The Hessian $\mathbf{H}_n(\boldsymbol{\beta})=\sum_iw_ib''(\mathbf{x}_i'\boldsymbol{\beta})\mathbf{x}_i\mathbf{x}_i'$ is non-random with $n^{-1}\mathbf{H}_n(\boldsymbol{\beta}_0)=\mathbf{G}_n$. For $|\boldsymbol{\beta}-\boldsymbol{\beta}_0|\leqslant1$ the arguments $\mathbf{x}_i'\boldsymbol{\beta}$ lie in $J=[-M(|\boldsymbol{\beta}_0|+1),M(|\boldsymbol{\beta}_0|+1)]$, where the continuous $b'''$ is bounded by some $K$ (the prerequisites' theorem **uniform continuity on compacts**); the mean value theorem gives $\|\mathbf{H}_n(\boldsymbol{\beta})-\mathbf{H}_n(\boldsymbol{\beta}_0)\|\leqslant nC_wKM^3|\boldsymbol{\beta}-\boldsymbol{\beta}_0|$. Parts (i)–(iii) of the theorem give the conclusions, with uniqueness from strict concavity once $\operatorname{rank}\mathbf{X}=k+1$, which $\mathbf{G}$ positive definite forces for large $n$ (two maximisers would give a strictly larger value at their midpoint). If $\operatorname{var} y_i=\phi v(\mu_i)/w_i=\phi b''(\mathbf{x}_i'\boldsymbol{\beta}_0)/w_i$, then $w_i^2\tau_i^2=\phi w_ib''(\mathbf{x}_i'\boldsymbol{\beta}_0)$ and $\mathbf{F}_n=\phi\mathbf{G}_n$. ∎

**Remark** (FREES §13.1, §13.3.1–13.3.2). Theorem 8.41 is FREES's statement that inference “only requires that we choose the mean and variance functions correctly”: consistency uses only the mean, and the model-based variance $\phi\mathbf{G}^{-1}$ also the variance function, the rest of the distribution being irrelevant. When $\operatorname{var} y_i=\sigma^2\phi v(\mu_i)/w_i$ with an extra factor $\sigma^2$ (FREES §13.3.2), $\mathbf{F}=\sigma^2\phi\mathbf{G}$ and the standard errors scale by $\sigma$, estimated by the Pearson statistic $\sum_iw_i(y_i-\hat\mu_i)^2/\{\phi v(\hat\mu_i)\}$ over $n-(k+1)$, an estimate proved consistent here for the Poisson family without weights (Proposition 8.17). FREES's “empirical” standard error in §13.9.3 (beyond the syllabus) inverts $\sum_i(\partial\mu_i/\partial\boldsymbol{\beta})(y_i-\mu_i)^{-2}(\partial\mu_i/\partial\boldsymbol{\beta}')$; the estimate of $\mathbf{G}^{-1}\mathbf{F}\mathbf{G}^{-1}$ is the sandwich $\hat{\mathbf{G}}^{-1}\hat{\mathbf{F}}\hat{\mathbf{G}}^{-1}$ with $(y_i-\hat\mu_i)^2$ in $\hat{\mathbf{F}}$, as in Proposition 8.20, which proves its consistency for the Poisson family.

**Example 8.12** (Ratemaking classification; FREES §13.2.2–13.3.1, Tables 13.3–13.4). FREES's `AutoCollision`: average severity of $8942$ UK collision claims in $32$ cells of eight driver age groups by four vehicle uses, fitted by a gamma model with logarithmic link and the claim counts as weights $w_i$, so that $\operatorname{var} y_i=\phi\mu_i^2/w_i$ for a cell average of $w_i$ claims:

In [ ]:
AC <- read.csv("data/frees/csv/AutoCollision.csv", quote = "", fileEncoding = "latin1")
AC$Age <- sub("\u0096", "-", AC$Age)
gA <- glm(Severity ~ factor(Age) + factor(Vehicle_Use), weights = Claim_Count,
          data = AC, family = Gamma(link = "log"))
cA <- coef(gA); levels(factor(AC$Vehicle_Use))
prem <- exp(cA[[1]]) * outer(exp(c(0, cA[2:8])), exp(c(cA[9:11], 0)))
dimnames(prem) <- list(levels(factor(AC$Age)), c("DriveLong", "DriveShort",
                                                "Pleasure", "Business"))
print(round(cbind(prem, Average = rowMeans(prem)), 2), digits = 7)
print(round(c(colMeans(prem), overall = mean(prem)), 2), digits = 7)

FREES's Table 13.4 is reproduced to the cent. Its first two columns are labelled “DriverShort” and “DriveLong” in that order; the script takes the coefficients in R's alphabetical order, Business (the base), DriveLong, DriveShort, Pleasure, so the column $322.17,\ldots$ is DriveLong and $265.56,\ldots$ is DriveShort, matching the raw averages of Table 13.3, where long commutes have the higher severities.

### 6.4 Goodness of fit

**Proposition 8.42** (The cross-product term; FREES §13.3.3). With $\hat y_i$ any fitted values, $\sum_i(y_i-\bar y)^2=\sum_i(y_i-\hat y_i)^2+\sum_i(\hat y_i-\bar y)^2+2\sum_i (y_i-\hat y_i)(\hat y_i-\bar y)$. The last sum is zero for least squares with an intercept; for a canonical-link generalised linear model with an intercept and $w_i=1$ it equals $\sum_i(y_i-\hat\mu_i)\hat\mu_i$, which need not vanish.

**Proof.** Expand $\{(y_i-\hat y_i)+(\hat y_i-\bar y)\}^2$. For least squares, $\mathbf{X}'\boldsymbol{\varepsilon}=\mathbf{0}$ makes $\boldsymbol{\varepsilon}$ orthogonal to $\hat{\mathbf{Y}}$ and to $\mathbf 1$. Under a canonical link $\sum_i(y_i-\hat\mu_i)=0$ by Proposition 8.37, which removes $\bar y$; the normal equations make $\mathbf{Y}-\hat{\boldsymbol{\mu}}$ orthogonal to $\mathbf{X}\mathbf b$, the vector of $\hat\eta_i$, not to the vector of $\hat\mu_i=b'(\hat\eta_i)$. ∎

**Definition 8.43** (Deviance; FREES §13.3.3). Let $\tilde\theta_i$ solve $b'(\tilde\theta_i)=y_i$ (the **saturated** fit) and let $\hat\theta_i$ be the fitted natural parameters. The **scaled deviance** is $D^*=2\{L(\tilde{\boldsymbol{\theta}})-L(\hat{\boldsymbol{\theta}})\}$ and the **deviance** is $D=\phi D^*$:

$$
D=2\sum_{i=1}^nw_i\bigl\{y_i(\tilde\theta_i-\hat\theta_i)-b(\tilde\theta_i)+b(\hat\theta_i)
\bigr\}=\sum_{i=1}^nd_i . \tag{16}
$$

**Proposition 8.44** (Non-negativity and special cases; FREES §13.3.3). Every $d_i\geqslant0$, with equality only if $\hat\mu_i=y_i$. With $w_i=1$ and the convention $0\ln0=0$,

$$
\begin{align*}
\text{normal:}\quad&D=\sum_i(y_i-\hat\mu_i)^2 ,\\
\text{Bernoulli:}\quad&D=2\sum_i\Bigl\{y_i\ln\frac{y_i}{\hat\mu_i}
                       +(1-y_i)\ln\frac{1-y_i}{1-\hat\mu_i}\Bigr\} ,\\
\text{Poisson:}\quad&D=2\sum_i\Bigl\{y_i\ln\frac{y_i}{\hat\mu_i}-(y_i-\hat\mu_i)\Bigr\} ,\\
\text{gamma:}\quad&D=2\sum_i\Bigl\{-\ln\frac{y_i}{\hat\mu_i}+\frac{y_i-\hat\mu_i}{\hat\mu_i}
                   \Bigr\} .
\end{align*}
$$

For $y_i$ at the boundary of the range of $b'$ ($0$ or $1$ for the Bernoulli, $0$ for the Poisson), $d_i$ is defined as the limit as $\tilde\theta_i\to\pm\infty$, which the convention gives.

**Proof.** With $h(\theta)=y_i\theta-b(\theta)$, $d_i=2w_i\{h(\tilde\theta_i)-h(\hat\theta_i)\}$; $h'=y_i-b'$ vanishes at $\tilde\theta_i$ and $h''=-b''<0$, so $h$ is strictly concave (the prerequisites' lemma **convexity from the derivative**) with its maximum at $\tilde\theta_i$. Normal: $b=\theta^2/2$, $\theta=\mu$, so $d_i=2\{y_i(y_i-\hat\mu_i)-y_i^2/2+\hat\mu_i^2/2\}=(y_i-\hat\mu_i)^2$. Poisson: $b=e^\theta$, $\theta=\ln\mu$, so $d_i=2\{y_i\ln y_i-y_i\ln\hat\mu_i-y_i+\hat\mu_i\}$; at $y_i=0$, $h(\theta)=-e^\theta\uparrow0$ as $\theta\to-\infty$, so $d_i=2\hat\mu_i$, the formula's value. Bernoulli: $b=\ln(1+e^\theta)$, $h(\theta)=y_i\ln\mu+(1-y_i)\ln(1-\mu)$ with $\mu=b'(\theta)$, whose supremum over $\mu$ is $y_i\ln y_i+(1-y_i)\ln(1-y_i)$, zero for $y_i\in\{0,1\}$. Gamma: $b=-\ln(-\theta)$, $\theta=-1/\mu$, so $d_i=2\{-1+y_i/\hat\mu_i-\ln y_i+\ln\hat\mu_i\}$. ∎

**Remark** (FREES §13.3.3). FREES's Bernoulli and Poisson deviances lack the factor $2$ of $D=\phi D^*$, and its Poisson term has $+(y_i-\hat\mu_i)$; with an intercept and the canonical link $\sum_i(y_i-\hat\mu_i)=0$, so that sign does not change $D$.

**Example 8.13** (The cross-product on `Bikeshare`; FREES §13.3.3).

In [ ]:
yB <- B$bikers; mB <- fitted(mpo)
round(c(cross_product = sum((yB - mB) * mB), total_SS = sum((yB - mean(yB))^2),
        R2_one_minus_error = 1 - sum((yB - mB)^2) / sum((yB - mean(yB))^2),
        R2_regression = sum((mB - mean(yB))^2) / sum((yB - mean(yB))^2)), 4)

The cross-product sum is $-7.77\times10^5$, so the two definitions of $R^2$ differ: $0.7338$ and $0.7439$.

## 7 Application: Medical Expenditures

> **Reading.** FREES §13.4. SRM learning outcomes 2(c), 2(d), 2(e).

**Proposition 8.45** (Gamma regression with logarithmic link). Let $\ln\mu_i=\mathbf{x}_i'\boldsymbol{\beta}$ and let $y_i$ be gamma with mean $\mu_i$ and shape $\nu$, so $\operatorname{var} y_i=\mu_i^2/\nu$. Then, with $\ell_n(\boldsymbol{\beta})=-\sum_i(y_i/\mu_i+\ln\mu_i)$,

(i) $L(\boldsymbol{\beta},\nu)=\nu\ell_n(\boldsymbol{\beta})+c(\nu,\mathbf{Y})$, so the maximiser $\mathbf b$ does not depend on $\nu$;

(ii) $\ell_n$ is concave, strictly when $\operatorname{rank}\mathbf{X}=k+1$, with $\nabla\ell_n=\sum_i(y_i/\mu_i-1)\mathbf{x}_i$ and $-\nabla^2\ell_n=\sum_i(y_i/\mu_i)\mathbf{x}_i\mathbf{x}_i'$;

(iii) the observed information for $\boldsymbol{\beta}$ is $\nu\sum_i(y_i/\mu_i)\mathbf{x}_i\mathbf{x}_i'$ and the expected information is $\nu\mathbf{X}'\mathbf{X}$; they differ because the link is not canonical.

**Proof.** The gamma density with shape $\nu$ and rate $\nu/\mu$ has $\ln f=\nu\ln\nu-\nu\ln\mu+(\nu-1)\ln y-\nu y/\mu-\ln\Gamma(\nu)$, which is (i). With $\mu_i=e^{\eta_i}$, $-(y_ie^{-\eta}+\eta)$ has derivatives $y_ie^{-\eta}-1$ and $-y_ie^{-\eta}<0$ in $\eta$, which with $\partial\eta_i/\partial\boldsymbol{\beta}=\mathbf{x}_i$ gives (ii) by the prerequisites' lemma **convexity from the derivative**; the Hessian is negative definite when the $\mathbf{x}_i$ span, as every $y_i>0$. Multiplying by $\nu$ gives the observed information, and $\operatorname{\mathsf{E}} y_i/\mu_i=1$ the expected one. The canonical link is $-1/\mu$ (Proposition 8.35). ∎

**Theorem 8.46** (Large-sample inference for gamma regression). Let the $y_i>0$ be independent with $\operatorname{\mathsf{E}} y_i=\mu_i=e^{\mathbf{x}_i'\boldsymbol{\beta}_0}$ and $\operatorname{var} y_i=\phi\mu_i^2$, let $|\mathbf{x}_i|\leqslant M$, $\sup_i\operatorname{\mathsf{E}}(y_i-\mu_i)^4<\infty$, and $n^{-1}\mathbf{X}'\mathbf{X}\to\mathbf{G}$ positive definite. Then $\mathbf b$ of Proposition 8.45 exists and is unique with probability tending to one, $\mathbf b\xrightarrow{\,p\,}\boldsymbol{\beta}_0$,

$$
\sqrt n(\mathbf b-\boldsymbol{\beta}_0)\xrightarrow{\,d\,} N(\mathbf{0},\phi\mathbf{G}^{-1}) ,
\qquad n^{-1}\sum_i(y_i/\hat\mu_i)\mathbf{x}_i\mathbf{x}_i'\xrightarrow{\,p\,}\mathbf{G} ,
$$

the Pearson estimate $\hat\phi=\{n-(k+1)\}^{-1}\sum_i(y_i/\hat\mu_i-1)^2$ satisfies $\hat\phi\xrightarrow{\,p\,}\phi$, and so standard errors from the observed and from the expected information, each with a consistent estimate of $\phi$, are both consistent.

**Proof.** Apply the prerequisites' theorem **concave maximum likelihood** to $\ell_n$, concave and smooth by Proposition 8.45. The means lie in $[e^{-M|\boldsymbol{\beta}_0|},e^{M|\boldsymbol{\beta}_0|}]$.

**Score.** $(y_i/\mu_i-1)\mathbf{x}_i$ has mean $\mathbf{0}$, variance $\phi\mathbf{x}_i\mathbf{x}_i'$, and fourth moments at most $M^4e^{4M|\boldsymbol{\beta}_0|}\sup_i\operatorname{\mathsf{E}}(y_i-\mu_i)^4$; so $\mathbf{F}_n=\phi n^{-1}\mathbf{X}'\mathbf{X}\to\phi\mathbf{G}$ and the Lyapunov corollary applies with fourth moments.

**Curvature.** $\mathbf{H}_n(\boldsymbol{\beta})=\sum_iy_ie^{-\mathbf{x}_i'\boldsymbol{\beta}}\mathbf{x}_i\mathbf{x}_i'$. Each entry of $n^{-1}\mathbf{H}_n(\boldsymbol{\beta}_0)-n^{-1}\mathbf{X}'\mathbf{X}$ is an average of independent mean-zero terms $(y_i/\mu_i-1)x_{ij}x_{il}$ of variance at most $\phi M^4$, so it tends to $0$ by Chebyshev's inequality, and $\mathbf{G}$ is the limit. For $|\boldsymbol{\beta}-\boldsymbol{\beta}_0|\leqslant1$, with $K=e^{M(|\boldsymbol{\beta}_0|+1)}$, the mean value theorem gives $|e^{-\mathbf{x}_i'\boldsymbol{\beta}}-e^{-\mathbf{x}_i'\boldsymbol{\beta}_0}|\leqslant KM|\boldsymbol{\beta}-\boldsymbol{\beta}_0|$, so $\|\mathbf{H}_n(\boldsymbol{\beta})-\mathbf{H}_n(\boldsymbol{\beta}_0)\|\leqslant nC_n|\boldsymbol{\beta}-\boldsymbol{\beta}_0|$ with $C_n=KM^3n^{-1}\sum_iy_i$. $\operatorname{\mathsf{E}} C_n\leqslant KM^3e^{M|\boldsymbol{\beta}_0|}$, so $\operatorname{\mathsf{P}}(C_n>c)\leqslant\operatorname{\mathsf{E}} C_n/c$ by Markov's inequality and $C_n$ is bounded in probability. This is the curvature condition.

Parts (i), (ii), (iii) and (v) of the theorem give existence, uniqueness (by Proposition 8.45(ii) $\ell_n$ is strictly concave once $\operatorname{rank}\mathbf{X}=k+1$, which $n^{-1}\mathbf{X}'\mathbf{X}\to\mathbf{G}$ positive definite forces for large $n$, and two maximisers would give a strictly larger value at their midpoint), consistency, the sandwich $\mathbf{G}^{-1}(\phi\mathbf{G})\mathbf{G}^{-1}=\phi\mathbf{G}^{-1}$, and the limit of the observed information; $n^{-1}\mathbf{X}'\mathbf{X}\to\mathbf{G}$ by assumption.

**Dispersion.** Follow the proof of Proposition 8.17 with $\mu_i^2$ in place of $\mu_i$. Write $e_i=y_i-\mu_i$, $\delta_i=\hat\mu_i-\mu_i$, $\Delta=\max_i|\delta_i|$, $m_-=e^{-M|\boldsymbol{\beta}_0|}$ and $m_+=e^{M|\boldsymbol{\beta}_0|}$. On $|\mathbf b-\boldsymbol{\beta}_0|\leqslant1$ the mean value theorem gives $\Delta\leqslant KM|\mathbf b-\boldsymbol{\beta}_0|$, so $\Delta\to0$ in probability. The average $n^{-1}\sum_ie_i^2/\mu_i^2$ has mean $\phi$ and variance at most $\sup_i\operatorname{\mathsf{E}} e_i^4/(nm_-^4)$, so it tends to $\phi$ in probability by Chebyshev's inequality. On $\{\Delta\leqslant m_-/2\}$, every $\hat\mu_i\geqslant m_-/2$, and

$$
\frac{(e_i-\delta_i)^2}{\hat\mu_i^2}-\frac{e_i^2}{\mu_i^2}
=\frac{e_i^2(\mu_i^2-\hat\mu_i^2)}{\mu_i^2\hat\mu_i^2}+\frac{\delta_i^2-2e_i\delta_i}{\hat\mu_i^2}
$$

with $|\mu_i^2-\hat\mu_i^2|\leqslant\Delta(2m_++\Delta)$. The average of these differences is therefore at most $4\Delta(2m_++\Delta)m_-^{-4}n^{-1}\sum_ie_i^2+4m_-^{-2}(\Delta^2+2\Delta n^{-1}\sum_i|e_i|)$ in absolute value, which tends to $0$ in probability as in Proposition 8.17; with $n/\{n-(k+1)\}\to1$, $\hat\phi\xrightarrow{\,p\,}\phi$. ∎

**Example 8.14** (Inpatient expenditures; FREES §13.4, Table 13.5). FREES's MEPS sample (Unit 7) restricted to the $157$ persons with positive inpatient expenditure `EXPENDIP`; race `OTHER` is merged into `WHITE`, as in Unit 7:

In [ ]:
H <- read.csv("data/frees/csv/HealthExpend.csv")
D <- subset(H, EXPENDIP > 0); D$RACE[D$RACE == "OTHER"] <- "WHITE"
D$RACE <- relevel(factor(D$RACE), ref = "WHITE")
D$REGION <- relevel(factor(D$REGION), ref = "WEST")
D$EDUC <- relevel(factor(D$EDUC), ref = "LHIGHSC")
D$PHSTAT <- relevel(factor(D$PHSTAT), ref = "EXCE")
D$INCOME <- relevel(factor(D$INCOME), ref = "HINCOME")
c(n = nrow(D), female_pct = round(100 * mean(D$GENDER), 1),
  median_female = median(D$EXPENDIP[D$GENDER == 1]),
  median_male = median(D$EXPENDIP[D$GENDER == 0]),
  median_all = median(D$EXPENDIP), maximum = max(D$EXPENDIP))
c(median_COUNTIP = median(D$COUNTIP), median_AGE = median(D$AGE))
do.call(rbind, lapply(c("RACE", "REGION", "EDUC", "PHSTAT", "MNHPOOR", "ANYLIMIT", "INCOME", "insure"),
  function(v) { g <- factor(D[[v]])
    data.frame(variable = v, level = levels(g), pct = round(100 * as.vector(prop.table(table(g))), 1),
               median = round(as.vector(tapply(D$EXPENDIP, g, median)))) }))

$72.0\%$ are women, with median $5546$ against $7313$ for men; the overall median is $5695$ and the largest expenditure $607{,}800.6$, FREES's figures. The medians by level of every other variable are FREES's Table 13.5 to the dollar (`OTHER` merged into `WHITE`, as in FREES's table); three race percentages differ from FREES's by $0.1$ to $0.2$ points (FREES prints $2.6$, $19.8$ and $75.6$ where the data give $2.5$, $19.7$ and $75.8$), a difference the merge does not explain. The poor self-rated physical health group stands out, with median $10{,}447$.

**Example 8.15** (Gamma and inverse Gaussian fits; FREES Table 13.6). By Proposition 8.45(i) the gamma $\mathbf b$ is `glm`'s, whatever $\nu$, and $\hat\nu$ maximises the log-likelihood at it; the same holds for the inverse Gaussian with $\phi$. FREES's $t$-values use the observed information of the joint fit in $(\boldsymbol{\beta},\ln\nu)$, computed numerically:

In [ ]:
suppressMessages(library(numDeriv))
ctl <- glm.control(epsilon = 1e-14, maxit = 500)
fit_ml <- function(g, ldens, lo, hi) {        # b from glm; scale by a 1-d search
  X <- model.matrix(g); yy <- g$y; b <- coef(g)
  ls <- optimize(function(s) sum(ldens(yy, fitted(g), exp(s))), c(lo, hi),
                 maximum = TRUE, tol = 1e-12)$maximum
  nll <- function(th) { k <- length(th)
    -sum(ldens(yy, exp(drop(X %*% th[-k])), exp(th[k]))) }
  th <- c(b, ls); V <- solve(hessian(nll, th)); k <- length(th)
  list(b = b, t = b / sqrt(diag(V)[-k]), scale = exp(ls), se_logscale = sqrt(V[k, k]),
       logLik = -nll(th), npar = k, X = X, y = yy) }
lgam <- function(y, mu, nu) dgamma(y, shape = nu, rate = nu / mu, log = TRUE)
lig <- function(y, mu, phi) statmod::dinvgauss(y, mean = mu, dispersion = phi, log = TRUE)
fF <- EXPENDIP ~ COUNTIP + AGE + GENDER + RACE + REGION + EDUC + PHSTAT + MNHPOOR +
  ANYLIMIT + INCOME + insure
D$BLACK <- 1 * (D$RACE == "BLACK"); D$POOR <- 1 * (D$PHSTAT == "POOR")
D$POORNEG <- 1 * (D$INCOME == "POOR")
fR <- EXPENDIP ~ COUNTIP + AGE + GENDER + BLACK + REGION + EDUC + POOR + MNHPOOR +
  ANYLIMIT + POORNEG
gF <- glm(fF, data = D, family = Gamma(link = "log"), control = ctl)
gR <- glm(fR, data = D, family = Gamma(link = "log"), control = ctl)
gI <- glm(fR, data = D, family = inverse.gaussian(link = "log"), start = coef(gR),
          control = ctl)
full <- fit_ml(gF, lgam, -3, 3); red <- fit_ml(gR, lgam, -3, 3)
ig <- fit_ml(gI, lig, -12, 0)
round(cbind(full_b = full$b, full_t = full$t), 3)
round(cbind(red_b = red$b, red_t = red$t, ig_b = ig$b, ig_t = ig$t), 3)
sm <- function(m, sc) c(scale = sc, scale_t = 1 / m$se_logscale, logLik = m$logLik,
  AIC = -2 * m$logLik + 2 * m$npar, AIC_frees = -2 * m$logLik + 2 * (m$npar - 1))
print(rbind(full = sm(full, full$scale), reduced = sm(red, red$scale),
            inv_gauss = sm(ig, sqrt(ig$scale)) * c(1, 2, 1, 1, 1)), digits = 7)

Every entry of FREES's Table 13.6 is reproduced. FREES's “Scale” is the shape $\hat\nu$ ($1.409$ and $1.280$), and for the inverse Gaussian it is $\sqrt{\hat\phi}=0.026$, whose $t$-value $17.72$ is twice that of $\hat\phi$. FREES's $\text{AIC}$ omits the scale parameter from the count: its $3163.34$ is $-2(-1558.67)+2\cdot23$ with the rounded log-likelihood, and counting the scale gives $3165.33$. The inverse Gaussian column is a fit to all $157$ observations, although the script fits that model with the largest expenditure removed, which would make its log-likelihood incomparable with the gamma's. The joint observed information is block-diagonal at the maximum, since $\partial^2L/\partial\boldsymbol{\beta}\,\partial\nu=\sum_i(y_i/\hat\mu_i-1)\mathbf{x}_i=\mathbf{0}$ by the score equations of Proposition 8.45(ii).

**Example 8.16** (Observed, expected and robust standard errors). For the full gamma model, the $t$-ratios of `COUNTIP` and `insure` from the observed information (FREES), the expected information with $\phi=1/\hat\nu$, R's `summary` (expected information with the Pearson $\hat\phi$), and the sandwich:

In [ ]:
X <- full$X; yy <- full$y; bF <- full$b; muF <- exp(drop(X %*% bF)); nu <- full$scale
Gobs <- nu * crossprod(X, (yy / muF) * X); Gexp <- nu * crossprod(X)
Fsw <- nu^2 * crossprod(X, (yy / muF - 1)^2 * X)
tt <- function(V) bF / sqrt(diag(V))
j <- c("COUNTIP", "insure")
round(rbind(observed = tt(solve(Gobs))[j], expected = tt(solve(Gexp))[j],
            glm_summary = coef(summary(gF))[j, 3],
            sandwich = tt(solve(Gexp, Fsw) %*% solve(Gexp))[j]), 3)
c(pearson_phi = summary(gF)$dispersion, one_over_nu = 1 / nu)

By Theorem 8.46 the third is consistent under the gamma model, and so are the first two provided $1/\hat\nu$ is a consistent estimate of $\phi$, which is not proved here; the consistency of the sandwich when the variance function is wrong is likewise not proved here for gamma regression (Proposition 8.20 treats the Poisson case). Here they range from $4.86$ to $7.49$ for `COUNTIP`. Proposition 8.45(iii) weights observation $i$ by $y_i/\hat\mu_i$ in the observed information, and a few very large expenditures dominate it.

## 8 Residuals

> **Reading.** FREES §13.5. SRM learning outcome 2(d).

**Definition 8.47** (Residuals; FREES §13.5). For a generalised linear model with fitted means $\hat\mu_i$:

(i) the **raw** residual is $y_i-\hat\mu_i$;

(ii) the **Pearson** residual is $r^P_i=\sqrt{w_i}(y_i-\hat\mu_i)/\sqrt{v(\hat\mu_i)}$;

(iii) the **deviance** residual is $r^D_i=\operatorname{sign}(y_i-\hat\mu_i)\sqrt{d_i}$, with $d_i$ of (16);

(iv) for a known transform $h$, $\{h(y_i)-\operatorname{\mathsf{E}} h(y_i)\}/\sqrt{\operatorname{var} h(y_i)}$ evaluated at $\hat\mu_i$; $h$ chosen to make $h(y_i)$ nearly normal gives the **Anscombe** residual, and $h$ chosen to make $\operatorname{var} h(y_i)$ nearly constant a **variance-stabilising** one, such as $2(\sqrt{y}-\sqrt{\mu})$ for the Poisson.

A **Cox–Snell** residual is $R(y_i;\mathbf{x}_i,\hat{\boldsymbol{\theta}})$ for a function $R$ making $R(y_i;\mathbf{x}_i,\boldsymbol{\theta})$ independent and identically distributed under the model; the Pearson residual at the true parameter is an example only up to its first two moments.

**Proposition 8.48** (Properties; FREES §13.5). At the true means, $\operatorname{\mathsf{E}} r^P_i=0$ and $\operatorname{var} r^P_i=\phi$. Always $\sum_i(r^P_i)^2=\sum_iw_i(y_i-\hat\mu_i)^2/v(\hat\mu_i)$, the Pearson statistic $\sum_iw_i(y_i-\hat\mu_i)^2/\{\phi v(\hat\mu_i)\}$ (remark after Theorem 8.41) times $\phi$, and $\sum_i(r^D_i)^2=D$.

**Proof.** $\operatorname{var} y_i=\phi v(\mu_i)/w_i$ (Definition 8.34); the sums are Definition 8.43 and the definitions above. ∎

**Example 8.17** (Transforms of the Poisson, gamma and binomial; FREES Table 13.7). Moments by summation over the Poisson and binomial mass functions, and by numerical integration for the gamma, for the square-root transform and FREES's Anscombe transform $1.5\lambda^{-1/6}\{y^{2/3}-(\lambda^{2/3}-\lambda^{-1/3}/9)\}$:

In [ ]:
mom <- function(lam) { j <- 0:qpois(1 - 1e-15, lam); pj <- dpois(j, lam)
  v <- function(h) sum((h - sum(h * pj))^2 * pj)
  an <- 1.5 * lam^(-1/6) * (j^(2/3) - (lam^(2/3) - lam^(-1/3) / 9))
  c(lambda = lam, var_2sqrt = v(2 * sqrt(j)), mean_anscombe = sum(an * pj),
    var_anscombe = v(an)) }
round(t(sapply(c(0.5, 1, 5, 20, 100), mom)), 4)
gam <- function(al) { a <- function(y) 3 * al^(1/6) * (y^(1/3) - (al^(1/3) - al^(-2/3) / 9))
  m <- integrate(function(y) a(y) * dgamma(y, al), 0, Inf, rel.tol = 1e-10)$value
  c(alpha = al, mean_anscombe = m,
    var_anscombe = integrate(function(y) (a(y) - m)^2 * dgamma(y, al), 0, Inf, rel.tol = 1e-10)$value) }
round(t(sapply(c(1, 2, 5, 20, 100), gam)), 4)
bin <- function(m, p) { y <- 0:m; py <- dbinom(y, m, p)
  h <- 2 * sqrt(m) * (asin(sqrt(y / m)) - asin(sqrt(p))); mu <- sum(h * py)
  c(m = m, p = p, mean_arcsine = mu, var_arcsine = sum((h - mu)^2 * py)) }
round(t(mapply(bin, c(5, 20, 100, 5, 20, 100), c(0.5, 0.5, 0.5, 0.1, 0.1, 0.1))), 4)

Both variances approach $1$ as $\lambda$ grows ($1.004$ and $1.002$ at $\lambda=100$), and neither is close to $1$ at $\lambda=1$ ($1.609$ and $1.184$). FREES's gamma row, $3\alpha^{1/6}\{(y/\gamma)^{1/3}-(\alpha^{1/3}-\alpha^{-2/3}/9)\}$, is evaluated with $\gamma=1$ (the transform is free of the scale): its mean is within $0.013$ of $0$ and its variance within $0.052$ of $1$ already at $\alpha=1$. For the binomial, the variance-stabilising arcsine transform $2\sqrt m\{\sin^{-1}\sqrt{y/m}-\sin^{-1}\sqrt p\}$ has variance near $1$ only for large $m$, and more slowly when $p$ is far from $\tfrac12$ ($1.04$ at $m=100$, $p=0.1$). For the gamma model of Example 8.15, Figure 8.1 plots the deviance residuals against normal quantiles.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.9)
gD <- glm(fR, data = D, family = Gamma(link = "log"))
op <- par(mfrow = c(1, 2), mar = c(4, 4.2, 1.6, 0.8), cex = 0.85)
plot(density(D$EXPENDIP[D$EXPENDIP < 6e5]), main = "kernel density",
     xlab = "EXPENDIP (largest omitted)", col = "#1E3A8A", lwd = 2)
qqnorm(residuals(gD, "deviance"), main = "deviance residuals", pch = 19, cex = 0.6,
       col = "grey30")
qqline(residuals(gD, "deviance"), col = "#B45309", lwd = 2)
par(op)

*Figure 8.1.* Left: kernel density of the positive inpatient expenditures without the largest (FREES Figure 13.1). Right: normal quantile plot of the deviance residuals of the reduced gamma model of Example 8.15.

## 9 The Tweedie Distribution

> **Reading.** FREES §13.6. SRM learning outcome 2(b).

**Lemma 8.49** (Sum of gammas). If $X\sim\mathrm{Gamma}(\alpha_1,\gamma)$ and $Y\sim\mathrm{Gamma}(\alpha_2,\gamma)$ are independent, with density $\gamma^\alpha u^{\alpha-1}e^{-\gamma u}/\Gamma(\alpha)$ on $(0,\infty)$, then $X+Y\sim\mathrm{Gamma}(\alpha_1+\alpha_2,\gamma)$.

**Proof.** First, $\Gamma(\alpha_1)\Gamma(\alpha_2)=\Gamma(\alpha_1+\alpha_2)B(\alpha_1,\alpha_2)$ with $B(\alpha_1,\alpha_2)=\int_0^1t^{\alpha_1-1}(1-t)^{\alpha_2-1}\mathrm{d} t$: writing the product as a double integral, substituting $u=st$, $v=s(1-t)$ with Jacobian $s$ and using Fubini–Tonelli,

$$
\int_0^\infty\!\!\int_0^\infty u^{\alpha_1-1}v^{\alpha_2-1}e^{-u-v}\mathrm{d} u\,\mathrm{d} v
=\int_0^\infty s^{\alpha_1+\alpha_2-1}e^{-s}\mathrm{d} s\int_0^1t^{\alpha_1-1}(1-t)^{\alpha_2-1}\mathrm{d} t .
$$

For $s>0$ the convolution density is $\gamma^{\alpha_1+\alpha_2}e^{-\gamma s}\{\Gamma(\alpha_1)\Gamma(\alpha_2)\}^{-1}\int_0^su^{\alpha_1-1} (s-u)^{\alpha_2-1}\mathrm{d} u$, and the substitution $u=st$ makes the integral $s^{\alpha_1+\alpha_2-1}B(\alpha_1,\alpha_2)$. ∎

**Theorem 8.50** (The Tweedie as a compound Poisson–gamma; FREES (13.9)–(13.12)). Let $N\sim\mathrm{Pois}(\lambda)$ and $Y_1,Y_2,\ldots$ independent $\mathrm{Gamma}(\alpha,\gamma)$, independent of $N$, and $T=\sum_{j\leqslant N}Y_j$, an empty sum being $0$. For $\mu>0$, $\phi>0$ and $1<p<2$ put

$$
\lambda=\frac{\mu^{2-p}}{\phi(2-p)} , \qquad \alpha=\frac{2-p}{p-1} , \qquad
\frac1\gamma=\phi(p-1)\mu^{p-1} . \tag{17}
$$

Then $\operatorname{\mathsf{P}}(T=0)=e^{-\lambda}$, $\operatorname{\mathsf{E}} T=\mu$, $\operatorname{var} T=\phi\mu^p$, and with respect to the measure $\delta_0+$ Lebesgue measure on $(0,\infty)$ (the measure of Definition 8.32, there called $\lambda$), $T$ has the density (12) with

$$
\theta=\frac{\mu^{1-p}}{1-p} , \quad b(\theta)=\frac{\mu^{2-p}}{2-p} , \quad
S(y,\phi)=\ln\sum_{n=1}^\infty\frac{1}{n!\,\Gamma(n\alpha)\,y}
\Bigl\{\frac{y^{\alpha}}{(p-1)^{\alpha}\phi^{1+\alpha}(2-p)}\Bigr\}^n\ (y>0) , \tag{18}
$$

and $S(0,\phi)=0$. So the Tweedie is a linear exponential family with $v(\mu)=\mu^p$.

**Proof.** Every $Y_j>0$, so $T=0$ exactly when $N=0$. By the prerequisites' corollary **random sums**, with $\operatorname{\mathsf{E}} Y_1=\alpha/\gamma$ and $\operatorname{var} Y_1=\alpha/\gamma^2$, $\operatorname{\mathsf{E}} T=\lambda\alpha/\gamma$ and $\operatorname{var} T=\lambda\alpha(1+\alpha)/\gamma^2$. Now $\alpha/\gamma=\phi(2-p)\mu^{p-1}$, so $\operatorname{\mathsf{E}} T=\mu$, and $1+\alpha=1/(p-1)$, so $\operatorname{var} T=\mu\cdot\phi(p-1)\mu^{p-1}/(p-1)=\phi\mu^p$.

Given $N=n\geqslant1$, $Y_1+\cdots+Y_n\sim\mathrm{Gamma}(n\alpha,\gamma)$ by Lemma 8.49 and induction, so for Borel $A\subset(0,\infty)$, by Fubini–Tonelli,

$$
\operatorname{\mathsf{P}}(T\in A)=\int_A\sum_{n\geqslant1}e^{-\lambda}\frac{\lambda^n}{n!}
\frac{\gamma^{n\alpha}}{\Gamma(n\alpha)}y^{n\alpha-1}e^{-\gamma y}\,\mathrm{d} y ,
$$

which is FREES (13.9). With (17), $\lambda+\gamma y=\{b(\theta)-y\theta\}/\phi$, since $\gamma y=y\mu^{1-p}/\{\phi(p-1)\} =-y\theta/\phi$ and $\lambda=b(\theta)/\phi$; and $\lambda\gamma^\alpha =\mu^{(2-p)-\alpha(p-1)}/\{\phi(2-p)(p-1)^\alpha\phi^\alpha\}$, where the exponent of $\mu$ is $(2-p)-(2-p)=0$. Hence the density is $\exp\{(y\theta-b(\theta))/\phi+S(y,\phi)\}$ with $S$ as in (18), and at $y=0$ the mass $e^{-\lambda}=e^{-b(\theta)/\phi}$ is the same expression with $S(0,\phi)=0$. As $\mu=\{(1-p)\theta\}^{1/(1-p)}$, $\mathrm{d}\theta/\mathrm{d}\mu=\mu^{-p}$, so $b'(\theta)=\mu^{1-p}\mu^p=\mu$ and $b''(\theta)=\mu^p$. ∎

**Remark** (FREES §13.6, exercise 13.3). $S(y,\phi)$ depends on $p$ through $\alpha$ but not on $\mu$, as FREES's exercise 13.3 requires. For $1<p<2$ the law has an atom at $0$ and a density on $(0,\infty)$ (Figure 8.2): a policy's aggregate loss, zero without a claim. The variance function $\mu^p$ is the Poisson's at $p=1$ and the gamma's at $p=2$ (FREES Table 13.1), and $1<p<2$ lies between them.

In [ ]:
suppressMessages(library(tweedie))
op <- par(mfrow = c(1, 2), mar = c(4, 4.2, 1.6, 0.8), cex = 0.85)
for (pp in c(1.3, 1.6)) {
  yv2 <- seq(0.02, 20, length.out = 400); lm2 <- 4^(2 - pp) / (2.5 * (2 - pp))
  plot(yv2, dtweedie(yv2, xi = pp, mu = 4, phi = 2.5), type = "l", lwd = 2,
       col = "#B45309", xlab = "y", ylab = "density", main = bquote(p == .(pp)),
       ylim = c(0, 0.26))
  segments(0, 0, 0, exp(-lm2), lwd = 3, col = "#1E3A8A")
  points(0, exp(-lm2), pch = 19, col = "#1E3A8A")
  text(1.0, exp(-lm2), sprintf("P(0) = %.2f", exp(-lm2)), pos = 4, cex = 0.8)
}
par(op)

*Figure 8.2.* Tweedie laws with $\mu=4$, $\phi=2.5$. The spike is the atom $e^{-\lambda}$ at $0$ of Theorem 8.50, the curve the density of the positive part.

## 10 Computation in R

> **Reading.** FREES chapters 12–13 scripts. SRM learning outcomes 2(d), 2(f). Beyond the syllabus: ISLR §4.7.7, the chapter's lab, which neither reading list includes.

### 10.1 The ISLR lab (beyond the syllabus)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
mod.lm <- lm(bikers ~ mnth + hr + workingday + temp + weathersit, data = Bikeshare)
c(feb_treatment = coef(mod.lm)[["mnthFeb"]], jan_sum_coding = coef(mlm)[["mnth1"]],
  sum_sq_prediction_diff = sum((predict(mod.lm) - predict(mlm))^2))
all.equal(predict(mod.lm), predict(mlm))
pl <- predict(mlm); pq <- predict(mpo, type = "response")
dec <- cut(pl, quantile(pl, 0:10 / 10), include.lowest = TRUE)
round(tapply(pq - pl, dec, mean), 1)

Under treatment coding the February coefficient $6.845$ is relative to January; under `contr.sum` the January coefficient $-46.087$ is relative to the average month, the twelve coefficients summing to zero (Figure 8.3). The fitted values are the same, as both codings span the same column space. By decile of the least squares prediction, the Poisson prediction exceeds it by $51.4$ in the lowest decile and $32.2$ in the highest, is within $0.1$ of it in the second and ninth, and falls below it in the third to eighth: ISLR's “larger for very low or very high levels of ridership”.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3.3)
cm <- function(m, i) { cc <- coef(m)[i]; c(cc, -sum(cc)) }
op <- par(mfrow = c(2, 2), mar = c(2.6, 3.4, 1.8, 0.6), mgp = c(1.9, 0.6, 0), cex = 0.7)
plot(cm(mlm, 2:12), type = "o", pch = 19, col = "#1E3A8A", xaxt = "n", xlab = "",
     ylab = "coefficient", main = "least squares: month")
axis(1, 1:12, c("J","F","M","A","M","J","J","A","S","O","N","D"))
plot(0:23, cm(mlm, 13:35), type = "o", pch = 19, col = "#1E3A8A", xlab = "",
     ylab = "coefficient", main = "least squares: hour")
plot(cm(mpo, 2:12), type = "o", pch = 19, col = "#B45309", xaxt = "n", xlab = "month",
     ylab = "coefficient", main = "Poisson: month")
axis(1, 1:12, c("J","F","M","A","M","J","J","A","S","O","N","D"))
plot(0:23, cm(mpo, 13:35), type = "o", pch = 19, col = "#B45309", xlab = "hour",
     ylab = "coefficient", main = "Poisson: hour")
par(op)

*Figure 8.3.* Month and hour coefficients under `contr.sum` coding, the last level being minus the sum of the others (ISLR Figures 4.13 and 4.15). Top: least squares, in bikers per hour. Bottom: Poisson, on the log scale.

### 10.2 Formulas against package output

**Example 8.18** (Poisson score, information and IRLS; Theorems 8.11,
8.40). With `glm`'s convergence tightened (Unit 7):

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
ctl <- glm.control(epsilon = 1e-14, maxit = 100)
m3t <- glm(fS, offset = LNWEIGHT, family = poisson, data = S, control = ctl)
X3 <- model.matrix(m3t)[, !is.na(coef(m3t))]; b3t <- coef(m3t)[!is.na(coef(m3t))]
mu3t <- fitted(m3t); off <- S$LNWEIGHT
irls <- function(X, y, off, link_inv, dmu, vfun, eta0) {
  eta <- eta0
  repeat { mu <- link_inv(eta); d <- dmu(eta); W <- d^2 / vfun(mu)
    z <- eta - off + (y - mu) / d
    b <- drop(solve(crossprod(X, W * X), crossprod(X, W * z)))
    eta_new <- drop(X %*% b) + off
    if (max(abs(eta_new - eta)) < 1e-13) return(b)
    eta <- eta_new } }
b_irls <- irls(X3, y, off, exp, exp, identity, log(y + 0.5))
signif(c(max_abs_score = max(abs(crossprod(X3, y - mu3t))),
         se_manual_minus_glm = max(abs(sqrt(diag(solve(crossprod(X3, mu3t * X3)))) -
                                       coef(summary(m3t))[, 2])),
         irls_minus_glm = max(abs(b_irls - b3t))), 3)

**Example 8.19** (A non-canonical link: gamma with logarithmic link). IRLS of Theorem 8.40 with $v(\mu)=\mu^2$ and $\mathrm{d}\mu/\mathrm{d}\eta=\mu$, so $W_i=1$ and $z_i=\eta_i+(y_i-\mu_i)/\mu_i$, against `glm`, and the score of Proposition 8.45(ii) at `glm`'s estimate:

In [ ]:
gRt <- glm(fR, data = D, family = Gamma(link = "log"), control = ctl)
XR <- model.matrix(gRt)
bR_irls <- irls(XR, D$EXPENDIP, 0, exp, exp, function(m) m^2, log(D$EXPENDIP))
signif(c(irls_minus_glm = max(abs(bR_irls - coef(gRt))),
         score_at_glm = max(abs(crossprod(XR, D$EXPENDIP / fitted(gRt) - 1)))), 3)

**Example 8.20** (Deviances and residuals; Propositions 8.44,
8.48).

In [ ]:
dP <- 2 * (ifelse(y == 0, 0, y * log(y / mu3t)) - (y - mu3t))
yg <- D$EXPENDIP; mg <- fitted(gRt); dG <- 2 * (-log(yg / mg) + (yg - mg) / mg)
signif(c(poisson_D_minus_glm = sum(dP) - deviance(m3t),
         gamma_D_minus_glm = sum(dG) - deviance(gRt),
         deviance_resid = max(abs(sign(y - mu3t) * sqrt(pmax(dP, 0)) -
                                  residuals(m3t, "deviance"))),
         pearson_resid = max(abs((y - mu3t) / sqrt(mu3t) - residuals(m3t, "pearson")))), 3)

**Example 8.21** (The deviance of a correct Poisson model; FREES §13.3.3). Intercept-only Poisson samples of size $n=2000$, $200$ replications for each mean:

In [ ]:
set.seed(21)
t(sapply(c(0.3, 1, 5, 50), function(m) {
  Dv <- replicate(200, deviance(glm(rpois(2000, m) ~ 1, family = poisson)))
  Xv <- replicate(200, { yy <- rpois(2000, m); sum((yy - mean(yy))^2) / mean(yy) })
  round(c(mu = m, mean_deviance = mean(Dv), mean_pearson = mean(Xv), n_minus_1 = 1999), 1) }))

The mean deviance is $1683$ at $\mu=0.3$ and $2296$ at $\mu=1$, against $1999$; the Pearson statistic stays within $10$ of $1999$ at every mean (Proposition 8.17).

## 11 Exercises

**Exercise 8.1** (FREES exercise 12.1). Show that the log-likelihood (2) has a maximum at $\hat\mu=\bar y$.

**Solution.** Proposition 8.3: when $\bar y>0$, $L'(\mu)=n(\bar y/\mu-1)$ is positive below $\bar y$ and negative above it, and $L''=-n\bar y/\mu^2<0$, so $\bar y$ is the unique maximiser. When every $y_i=0$, $L(\mu)=-n\mu$ has supremum $0$ at the excluded $\mu=0$ and no maximiser in $(0,\infty)$. ■

**Exercise 8.2** (FREES exercise 12.2). For the data of FREES Table 12.1, confirm that the Pearson statistic is $41.98$.

**Solution.** Example 8.2 computes $41.984$ from FREES's five cells $\{0\},\ldots,\{4\}$ and the fitted counts $n\,e^{-\bar y}\bar y^j/j!$: the cells contribute $(6996-6977.86)^2/6977.86=0.05$, $(455-487.69)^2/487.69=2.19$, $(28-17.043)^2/17.043=7.04$, $(4-0.3971)^2/0.3971=32.69$ and $0.0069$. Cell $\{3\}$ supplies $78\%$ of the total. ■

**Exercise 8.3** (FREES exercise 12.3). In a Poisson regression with an intercept, let $e_i=y_i-\hat\mu_i$. Show that (a) the average residual is $0$; (b) the correlation between the residuals and each explanatory variable is $0$.

**Solution.** (a) Corollary 8.12 with $x_{i0}=1$: $\sum_ie_i=0$.

(b) For each $j$, $\sum_ix_{ij}e_i=0$ by the same corollary, so the sample covariance $n^{-1}\sum_i(x_{ij}-\bar x_j)(e_i-\bar e)=n^{-1}\sum_ix_{ij}e_i-\bar x_j\bar e=0$ by (a). The correlation is this covariance divided by the two standard deviations, hence $0$ whenever both are positive, and undefined otherwise. Example 8.18 prints $\max_j|\sum_ix_{ij}e_i|$ at the level of rounding error. ■

**Exercise 8.4** (FREES exercise 12.4). (a) Let $y_1,\ldots,y_n$ be independent $\mathrm{NB}(r,p)$. Determine the maximum likelihood estimators. (b) Do the same in the parametrisation $\sigma=1/r$, $\mu=r(1-p)/p$. (c) For independent $y_i\sim\mathrm{NB}(\mu_i,\sigma)$ with $\mu_i=\exp(\mathbf{x}_i'\boldsymbol{\beta})$, determine the score in $\sigma$ and $\boldsymbol{\beta}$.

**Solution.** (a) Assume $\bar y>0$; if every $y_i=0$, the $p$-derivative $nr/p$ is positive on $(0,1)$ and no maximiser exists. With $N_r(y)=\ln\Gamma(y+r)-\ln\Gamma(r)-\ln y!$, the log-likelihood is $\sum_iN_r(y_i)+nr\ln p+n\bar y\ln(1-p)$. For fixed $r$ its $p$-derivative $nr/p-n\bar y/(1-p)$ is strictly decreasing and vanishes at $\hat p(r)=r/(r+\bar y)$, the maximiser in $p$. Substituting, the profile log-likelihood is

$$
\ell_p(r)=\sum_iN_r(y_i)+nr\ln\frac{r}{r+\bar y}+n\bar y\ln\frac{\bar y}{r+\bar y} ,
$$

and since $\Gamma(y+r)/\Gamma(r)=\prod_{l=0}^{y-1}(r+l)$,

$$
\ell_p'(r)=\sum_i\sum_{l=0}^{y_i-1}\frac1{r+l}+n\ln\frac{r}{r+\bar y} .
$$

$\hat r$ is a root of $\ell_p'$, found numerically (the root with the largest $\ell_p$, provided that value exceeds the Poisson log-likelihood at $\bar y$; otherwise none exists), and $\hat p=\hat r/(\hat r+\bar y)$. If some $y_i\geqslant1$, $\ell_p(r)\to-\infty$ as $r\downarrow0$, because $\ln\prod_l(r+l)$ contains $\ln r$ and the other terms stay bounded. So if $\ell_p'$ has no root it is positive throughout, $\ell_p$ increases, and its supremum is the limit $r\to\infty$, the Poisson log-likelihood at $\bar y$ (Corollary 8.23); no maximiser exists.

(b) The map $(r,p)\mapsto(\sigma,\mu)=(1/r,r(1-p)/p)$ is a bijection of $(0,\infty)\times(0,1)$ onto $(0,\infty)^2$, so the maximiser maps to the maximiser: $\hat\sigma=1/\hat r$ and $\hat\mu=\hat r(1-\hat p)/\hat p=\hat r\cdot\bar y/\hat r=\bar y$.

(c) With $r=1/\sigma$, $\ln f(y)=\ln\Gamma(y+1/\sigma)-\ln\Gamma(1/\sigma)-\ln y!-\sigma^{-1}\ln(1+\sigma\mu) +y\ln(\sigma\mu)-y\ln(1+\sigma\mu)$. Then $\partial\ln f/\partial\mu=(y-\mu)/\{\mu(1+\sigma\mu)\}$, and with $\partial\mu_i/\partial\boldsymbol{\beta}=\mu_i\mathbf{x}_i$,

$$
\frac{\partial L}{\partial\boldsymbol{\beta}}=\sum_i\frac{y_i-\mu_i}{1+\sigma\mu_i}\mathbf{x}_i ,
\qquad
\frac{\partial L}{\partial\sigma}=\sum_i\Bigl[\frac{1}{\sigma^2}\Bigl\{\ln(1+\sigma\mu_i)
-\sum_{l=0}^{y_i-1}\frac{1}{1/\sigma+l}\Bigr\}+\frac{y_i-\mu_i}{\sigma(1+\sigma\mu_i)}\Bigr] ,
$$

the last term collecting $-\mu_i/\{\sigma(1+\sigma\mu_i)\}+y_i/\sigma-y_i\mu_i/(1+\sigma\mu_i)$. Both vanish at the `glm.nb` fit of Example 8.10, and (a) on the Singapore counts without covariates: ■

In [ ]:
mn <- fitted(nb3); sg <- 1 / nb3$theta; Xn <- model.matrix(nb3)
inner <- sapply(y, function(k) if (k == 0) 0 else sum(1 / (1 / sg + 0:(k - 1))))
dsig <- sum((log(1 + sg * mn) - inner) / sg^2 + (y - mn) / (sg * (1 + sg * mn)))
dprof <- function(r) sum(sapply(y, function(k) if (k == 0) 0 else sum(1 / (r + 0:(k - 1))))) +
  n * log(r / (r + ybar))
r_hat <- uniroot(dprof, c(0.01, 100))$root
signif(c(score_beta = max(abs(crossprod(Xn, (y - mn) / (1 + sg * mn)))),
         score_sigma = dsig, r_hat = r_hat, p_hat = r_hat / (r_hat + ybar),
         glm_nb_r = glm.nb(y ~ 1)$theta), 4)

**Exercise 8.5** (FREES exercise 12.5). For the MEPS data, with $y=$ `COUNTOP`, the number of outpatient visits, and the explanatory variables of FREES §11.4: (a) tabulate and summarise $y$ and compare its mean and variance; (b) tabulate its means by the levels of each explanatory variable; (c) fit a Poisson model without covariates and compute the individual-level Pearson statistic; (d) fit the Poisson model with the covariates: (i) significance, (ii) the `GENDER` coefficient, (iii) the Pearson statistic and a choice between the models, (iv) the quasi-likelihood refit; (e) fit the negative binomial: (i) significance, (ii) the Pearson statistic and $\text{AIC}$, (iii) a likelihood ratio test for income; (f) fit a logistic regression for $\{y>0\}$ and compare it with (e).

**Solution.** FREES's parts (d)(iii) and (e)(ii) refer to the Pearson statistic of “part (b)”; it is part (c).

In [ ]:
H$RACE[H$RACE == "OTHER"] <- "WHITE"; H$RACE <- relevel(factor(H$RACE), ref = "WHITE")
H$REGION <- relevel(factor(H$REGION), ref = "WEST")
H$EDUC <- relevel(factor(H$EDUC), ref = "LHIGHSC")
H$PHSTAT <- relevel(factor(H$PHSTAT), ref = "EXCE")
H$INCOME <- relevel(factor(H$INCOME), ref = "HINCOME")
yo <- H$COUNTOP; table(cut(yo, c(-1, 0, 1, 2, 5, 10, 20, 50, 200)))
c(mean = mean(yo), variance = var(yo), median = median(yo), max = max(yo))
sapply(c("GENDER", "RACE", "REGION", "EDUC", "PHSTAT", "MNHPOOR", "ANYLIMIT",
         "INCOME", "insure"), function(v) round(tapply(yo, H[[v]], mean), 2))

(a) $648$ of the $2000$ persons have no visit, the median is $2$ and the maximum $167$: a long right tail. The variance $156.3$ is $27.5$ times the mean $5.67$, far from equidispersion. (b) The means differ sharply by physical health ($3.19$ excellent, $17.40$ poor), activity limitation ($3.57$ against $13.01$), mental health, insurance ($2.84$ against $6.49$) and sex ($4.25$ men, $6.95$ women), and less by region and income.

In [ ]:
fO <- COUNTOP ~ GENDER + RACE + REGION + EDUC + PHSTAT + MNHPOOR + ANYLIMIT + INCOME + insure
pearson <- function(m, v) sum((yo - fitted(m))^2 / v(fitted(m)))
p0 <- glm(COUNTOP ~ 1, family = poisson, data = H); p1 <- glm(fO, family = poisson, data = H)
q1 <- glm(fO, family = quasipoisson, data = H); n1 <- glm.nb(fO, data = H)
n2 <- glm.nb(update(fO, . ~ . - INCOME), data = H)
n2r <- glm(update(fO, . ~ . - INCOME), family = negative.binomial(n1$theta), data = H)
l1 <- glm(update(fO, I(COUNTOP > 0) ~ .), family = binomial, data = H)
round(cbind(pois_b = coef(p1), pois_z = coef(summary(p1))[, 3],
            quasi_t = coef(summary(q1))[, 3], nb_b = coef(n1),
            nb_z = coef(summary(n1))[, 3], logit_z = coef(summary(l1))[, 3]), 2)
LRi <- 2 * (as.numeric(logLik(n1)) - as.numeric(logLik(n2)))
LRr <- 2 * (as.numeric(logLik(n1)) - as.numeric(logLik(n2r)))
round(c(X2_null = pearson(p0, identity), X2_poisson = pearson(p1, identity),
        X2_negbin = pearson(n1, function(m) m + m^2 / n1$theta), phi_quasi = summary(q1)$dispersion,
        AIC_null = AIC(p0), AIC_poisson = AIC(p1), AIC_negbin = AIC(n1), r_negbin = n1$theta,
        exp_gender = exp(coef(p1)[["GENDER"]]), LR_income = LRi,
        p_income = pchisq(LRi, 4, lower.tail = FALSE), LR_income_fixed_r = LRr,
        p_income_fixed_r = pchisq(LRr, 4, lower.tail = FALSE)), 3)

(c) The Pearson statistic of the null Poisson model is $55{,}044$ on $1999$ degrees of freedom.

(d)(i) Every coefficient but the intercept, `BLACK` ($z=1.96$ to two decimals, $1.959$), `NORTHEAST` and `NPOOR` has $|z|>1.96$, twelve of them $|z|>8$. (ii) Women have $e^{0.420}=1.52$ times the expected visits of comparable men (Proposition 8.10(ii)). (iii) The Pearson statistic falls to $32{,}398$ on $1979$ degrees of freedom, $\hat\phi=16.4$; the covariates help, but by Proposition 8.17 the Poisson variance is wrong by a factor of about $16$, and the $z$ values of (i) are not to be trusted. (iv) Under Theorem 8.19 every $t$ is the $z$ divided by $\sqrt{16.37}=4.05$: `SOUTH`, `LINCOME`, `MINCOME`, income `POOR` and `NATIVE` are no longer significant at $5\%$ (`BLACK`, with $|z|<1.96$, was not significant in (i)), while `GENDER`, health, `ANYLIMIT`, education, `insure`, `ASIAN`, `MIDWEST` and `MNHPOOR` remain so.

(e)(i) With $\hat r=0.562$ the significant variables are `GENDER`, `ASIAN`, `MIDWEST`, education, all health levels, `MNHPOOR`, `ANYLIMIT`, `LINCOME` and `insure`. (ii) The Pearson statistic with $v(\mu)=\mu+\mu^2/\hat r$ is $3678$; the $\text{AIC}$ falls from $29{,}853$ (null) and $22{,}646$ (Poisson) to $9992$, so the negative binomial is preferred. (iii) Dropping the four income indicators gives $\text{LRT}=18.96$, $p=0.0008$ against $\chi^2_4$, with $r$ re-estimated in both fits by `glm.nb`; holding $r$ at $\hat r$ in both fits gives $\text{LRT}=19.04$, again $p=0.0008$. The prerequisites' theorem **likelihood ratio** covers the second version for a known $r$, under $|\mathbf{x}_i|\leqslant M$ and $n^{-1}\sum_i\mu_i\mathbf{x}_i\mathbf{x}_i'/(1+\sigma\mu_i)\to\mathbf{F}$ positive definite. Its (L1) is Proposition 8.25. The score $(y_i-\mu_i)\mathbf{x}_i/(1+\sigma\mu_i)$ has mean $\mathbf{0}$, variance $\mu_i\mathbf{x}_i\mathbf{x}_i'/(1+\sigma\mu_i)$ by (9), and bounded fourth moments since the $\mu_i$ are bounded, so the Lyapunov corollary gives (L2). The Hessian $\sum_i\{(y_i+r)r\mu_i/(r+\mu_i)^2\}\mathbf{x}_i\mathbf{x}_i'$ has mean $\sum_i\mu_i\mathbf{x}_i\mathbf{x}_i'/(1+\sigma\mu_i)$, so $\mathbf{G}=\mathbf{F}$, and converges after division by $n$ by Chebyshev's inequality; since $r e^\eta/(r+e^\eta)^2\leqslant\frac14$, the derivative of $(y_i+r)re^\eta/(r+e^\eta)^2$ in $\eta$ is at most $(y_i+r)/4$ in absolute value, so the curvature condition holds with $L_n=M^3n^{-1}\sum_i(y_i+r)/4$, bounded in probability by Markov's inequality as in Theorem 8.46; this is (L3). The restricted model satisfies the same conditions. Neither the re-estimation of $r$ nor the use of the estimate $\hat r$ in place of a known $r$ is covered here, so both $p$-values are approximations; income is significant by either.

(f) Every slope of the logistic regression of $\{y>0\}$ has the sign of the negative binomial's except `SOUTH` ($z=1.47$ against $-0.84$), insignificant in both. Significance differs: `ASIAN`, `MNHPOOR` and `HIGHSCH` matter for the number of visits and not for whether any occurs, `MINCOME` and income `POOR` the reverse. ■

**Exercise 8.6** (FREES exercise 12.6). Let $x_i\in\{0,1\}$, with $n_0$ observations at $x=0$ of mean $\bar y_0$ and $n_1=n-n_0$ at $x=1$ of mean $\bar y_1$, and fit $\ln\mu_i=\beta_0+\beta_1x_i$. (i) Find the maximum likelihood estimators. (ii) Evaluate them for $n_0=10$, $n_1=90$, $\bar y_0=0.20$, $\bar y_1=0.05$. (iii) Find the information matrix.

**Solution.** (i) By Corollary 8.12 the score equations are $\sum_i(y_i-\mu_i)=0$ and $\sum_ix_i(y_i-\mu_i)=0$, that is $n_1e^{\beta_0+\beta_1}=n_1\bar y_1$ and $n_0e^{\beta_0}=n_0\bar y_0$. When $\bar y_0,\bar y_1>0$, $b_0=\ln\bar y_0$ and $b_1=\ln(\bar y_1/\bar y_0)$, the unique maximiser by the strict concavity of Theorem 8.11 and the prerequisites' proposition **first-order condition**. If $\bar y_1=0$ no maximiser exists by Corollary 8.14; if $\bar y_0=0$, by Theorem 8.13(i) with $\mathbf{c}=(-1,1)'$, for which $\mathbf{x}_i'\mathbf{c}=-1$ at $x_i=0$ and $0$ at $x_i=1$. (ii) $b_0=\ln0.20=-1.609$ and $b_1=\ln0.25=-1.386$. (iii) By Theorem 8.11, with $\mu_0=e^{\beta_0}$ and $\mu_1=e^{\beta_0+\beta_1}$,

$$
\mathcal{I}(\boldsymbol{\beta})=\begin{pmatrix}n_0\mu_0+n_1\mu_1 & n_1\mu_1\\ n_1\mu_1 & n_1\mu_1\end{pmatrix} ,
\qquad
\mathcal{I}(\mathbf b)=\begin{pmatrix}6.5&4.5\\4.5&4.5\end{pmatrix}
$$

for the data of (ii), since $n_0\bar y_0=2$ and $n_1\bar y_1=4.5$. ■

**Exercise 8.7** (FREES exercise 13.1). Verify the gamma entry of FREES Table 13.8: show that the gamma is a linear exponential family, give $(\theta,\phi,b(\theta),S(y,\phi))$ in terms of $(\alpha,\gamma)$, and check the mean and variance.

**Solution.** Proposition 8.35: $\theta=-\gamma/\alpha$, $\phi=1/\alpha$, $b(\theta)=-\ln(-\theta)$ on $\theta<0$, and $S(y,\phi)=-\phi^{-1}\ln\phi-\ln\Gamma(\phi^{-1})+(\phi^{-1}-1)\ln y$. By Theorem 8.33, $\operatorname{\mathsf{E}} y=b'(\theta)=-1/\theta=\alpha/\gamma$ and $\operatorname{var} y=\phi b''(\theta)=\phi/\theta^2=\alpha/\gamma^2$, the gamma's moments ($\operatorname{\mathsf{E}} y=\int_0^\infty\gamma^\alpha y^\alpha e^{-\gamma y}\mathrm{d} y/\Gamma(\alpha) =\Gamma(\alpha+1)/\{\gamma\Gamma(\alpha)\}=\alpha/\gamma$, and likewise $\operatorname{\mathsf{E}} y^2=\alpha(\alpha+1)/\gamma^2$). FREES's table has $+\phi^{-1}\ln\phi$: ■

In [ ]:
al <- 2.5; ga <- 0.7; yv <- c(0.5, 2, 6); th <- -ga / al; ph <- 1 / al
lef <- function(sgn) exp((yv * th + log(-th)) / ph + sgn * log(ph) / ph - lgamma(1 / ph) +
                         (1 / ph - 1) * log(yv))
signif(rbind(dgamma = dgamma(yv, al, ga), corrected = lef(-1), frees_table = lef(+1)), 5)

**Exercise 8.8** (FREES exercise 13.2). For the `AutoCollision` data, fit a gamma regression with logarithmic link and claim counts as weights on age group and vehicle use, and verify FREES Table 13.4.

**Solution.** Example 8.12: the fitted cell means $\hat\mu_{ij}=e^{b_0}e^{\hat\alpha_i}e^{\hat\tau_j}$ (Proposition 8.10 (iv), which needs only the logarithmic link) reproduce every entry of Table 13.4, whose first two column labels are interchanged. With claim counts as weights, a cell average of $w_i$ independent claims of variance $\phi\mu_i^2$ has variance $\phi\mu_i^2/w_i$, the weighting of Definition 8.34. ■

**Exercise 8.9** (FREES exercise 13.3). Verify that the Tweedie distribution is a linear exponential family by giving $S(y,\phi)$, and verify the moments $\operatorname{\mathsf{E}} T=\mu$ and $\operatorname{var} T=\phi\mu^p$, $T$ as in Theorem 8.50.

**Solution.** Theorem 8.50 and (18). The series against `tweedie::dtweedie` at $\mu=4$, $\phi=2.5$, $p=1.6$, and the moments by simulation: ■

In [ ]:
p0 <- 1.6; mu0 <- 4; phi0 <- 2.5; al <- (2 - p0) / (p0 - 1)
th <- mu0^(1 - p0) / (1 - p0); bt <- mu0^(2 - p0) / (2 - p0)
S_tw <- function(u) log(sum(exp((1:300) * log(u^al / ((p0 - 1)^al * phi0^(1 + al) *
  (2 - p0))) - lgamma(2:301) - lgamma((1:300) * al) - log(u))))
dens <- sapply(yv, function(u) exp((u * th - bt) / phi0 + S_tw(u)))
signif(rbind(series = dens, dtweedie = dtweedie(yv, xi = p0, mu = mu0, phi = phi0)), 7)
set.seed(11); lam <- bt / phi0; gam <- 1 / (phi0 * (p0 - 1) * mu0^(p0 - 1))
Ss <- vapply(rpois(2e5, lam), function(k) if (k == 0) 0 else
  sum(rgamma(k, shape = al, rate = gam)), numeric(1))
round(c(mean = mean(Ss), theory = mu0, var = var(Ss), theory_var = phi0 * mu0^p0,
        P0 = mean(Ss == 0), theory_P0 = exp(-lam)), 3)

**Exercise 8.10** (SOA sample question 7). Which pair of distribution and link function is most appropriate to model whether a person is hospitalised? (A) normal, identity; (B) normal, logit; (C) binomial, linear; (D) binomial, logit; (E) it cannot be determined from the information given.

**Solution.** The response is binary, so its law is Bernoulli, the binomial with $m=1$ (Unit 7; Proposition 8.35); the normal (A, B) puts mass on every interval and cannot be the law of a $\{0,1\}$ variable. The mean is a probability, and of the two links only the logit maps $(0,1)$ onto $\mathbb{R}$, so that every $\mathbf{x}'\boldsymbol{\beta}$ gives a mean in $(0,1)$; a linear link does not (Proposition 8.8(i)'s argument, as in Unit 7's linear probability model). The answer is (D). ■

**Exercise 8.11** (SOA sample question 20). A simple linear model for a probability gives predicted values below $0$ and above $1$. Which is the most appropriate remedy? (A) Limit the data to observations expected to give predictions in $[0,1]$. (B) Treat predictions below $0$ as $0$ and above $1$ as $1$. (C) Use a logit function to transform the linear model into only predicting values between $0$ and $1$. (D) Use the canonical link of the Poisson distribution to transform the linear model into only predicting values between $0$ and $1$. (E) None of the above.

**Solution.** (A) selects observations by their covariates and still leaves a linear mean that leaves $[0,1]$ for other covariate values (Unit 7, the first drawback); (B) changes the fitted values without changing the model, which still implies negative variances $\hat y(1-\hat y)$ outside $[0,1]$. (C) is the logistic model: the inverse logit maps $\mathbb{R}$ onto $(0,1)$. (D) The Poisson's canonical link is $\ln\mu$ (Proposition 8.35), whose inverse $e^\eta$ maps $\mathbb{R}$ onto $(0,\infty)$, not into $(0,1)$. The answer is (C). ■

**Exercise 8.12** (SOA sample question 45). A generalised linear model with logarithmic link for an exam score on a $0$–$10$ scale has intercept $-0.1$, study time (in units of $100$ hours) $0.5$, first attempt $0.5$, master's degree $-0.1$, and attempt $\times$ master's interaction $0.2$. An applicant with a master's degree, for whom the exam would be a first attempt, is to be offered half of the study time that gives an expected score of $6.0$. Calculate the study time offered: (A) 123 hours; (B) 126; (C) 129; (D) 132; (E) 135.

**Solution.** With study time $s$ (in hundreds of hours), first attempt $1$, master's $1$ and the interaction $1$, the linear predictor is $-0.1+0.5s+0.5-0.1+0.2=0.5+0.5s$, and the logarithmic link gives $\mu=e^{0.5+0.5s}$. Setting $\mu=6$, $0.5s=\ln6-0.5=1.29176$, so $s=2.5835$, that is $258.35$ hours, and half of it is $129.2$ hours: (C). ■

**Exercise 8.13** (SOA sample question 52). Which statements about Pearson residuals are true? I. They can be used to calculate a goodness-of-fit statistic. II. They can be used to detect if additional variables of interest can improve the model specification. III. They can be used to identify unusual observations. (A) I only; (B) II only; (C) III only; (D) I, II and III; (E) none of (A)–(D).

**Solution.** I: $\sum_ir_i^2$ is the Pearson statistic (Definition 8.16), whose ratio to $n-(k+1)$ tends to $1$ under the model and to $\phi$ under (6) (Proposition 8.17). II and III: at the true means each $r_i$ has mean $0$ and variance $\phi$ (Proposition 8.48), the same for every observation, so a residual far from $0$ marks an unusual observation, and a trend of the $r_i$ against a candidate covariate is structure the mean has missed; the raw residuals, of variance $\phi v(\mu_i)/w_i$, mix such a trend with the variance function. All three are true: (D). ■

**Exercise 8.14** (SOA sample question 75). Which statement about the Tweedie distribution with $1<p<2$ is NOT true? (A) Its variance is $\phi\mu^p$. (B) The probability mass at zero is $e^{-\lambda}$. (C) It is a member of the linear exponential family. (D) The Poisson distribution is a special case of this Tweedie distribution. (E) This Tweedie distribution is a Poisson sum of gamma random variables.

**Solution.** (A), (B), (C) and (E) are Theorem 8.50. For (D): with $1<p<2$, $\operatorname{\mathsf{P}}(T>0)=1-e^{-\lambda}>0$ and the law of $T$ on $(0,\infty)$ has a density, so $\operatorname{\mathsf{P}}(T\in A)=0$ for the countable set $A=\{1,2,\ldots\}$ while $\operatorname{\mathsf{P}}(T>0)>0$; a Poisson variable puts all its positive mass on $A$. So no member with $1<p<2$ is a Poisson, and (D) is the false statement.

As $p\downarrow1$ with $\mu,\phi$ fixed the law tends to that of $\phi N$ with $N\sim\mathrm{Pois}(\mu/\phi)$: in (17), $\lambda\to\mu/\phi$, and each $Y_j$ has mean $\alpha/\gamma=\phi(2-p)\mu^{p-1}\to\phi$ and variance $(\alpha/\gamma)^2/\alpha\to0$, so $\operatorname{\mathsf{E}}|Y_j-\phi|\leqslant\{\operatorname{var} Y_j+(\operatorname{\mathsf{E}} Y_j-\phi)^2\}^{1/2}\to0$. Conditioning on $N$, $\operatorname{\mathsf{E}}|T-\phi N|\leqslant\operatorname{\mathsf{E}} N\cdot\operatorname{\mathsf{E}}|Y_1-\phi|=\lambda\operatorname{\mathsf{E}} |Y_1-\phi|\to0$, while $\phi N\xrightarrow{\,d\,}\phi\,\mathrm{Pois}(\mu/\phi)$ because the Poisson probabilities are continuous in $\lambda$; Slutsky's theorem gives the limit. It is a Poisson only when $\phi=1$, a limit and not a member. ■